# bronze → HTML: 数字で見る秋田(一般の方向け)

`opendata_bronze_report.ipynb` が「機械で読めるか」を見るエンジニア向けなのに対して、こちらは
**秋田のことを知ってもらう** ための読み物。秋田県のオープンデータ(bronze)から、人口・クマ・エネルギー・落とし物など
暮らしに近い数字と、地震の被害想定(250m メッシュ)を選んで図にする。

SageMaker Unified Studio の JupyterLab(Python 3 カーネル)で、`opendata_landing_to_bronze.ipynb` のあとに上から順に実行する。
bronze を **Athena を通さず直接** 読む。

| 出力 | 場所 |
| --- | --- |
| HTML | `s3://<bucket>/akita/reports/akita/index.html` |

外部ファイルを一切参照しない HTML 1枚(図は SVG を埋め込み、JavaScript なし)。公開する前提なので、
HTML にはバケット名・アカウント ID を入れない。公開用のバケットへのコピーはこのノートブックの外で行う。

実行するときにインターネットから2つ取ってくる(取れなければ、その図だけ地図なし・位置なしで続ける)。
国土地理院の地理院タイル(地図の背景)と、国土交通省の大字・町丁目レベル位置参照情報(学校・保育施設の住所から位置を決める)。
どちらも HTML に埋め込むか集計に使うだけで、ページを開いた人のブラウザが外へ読みに行くことはない。

図にするデータは決め打ち(データセット ID とリソース名)で選ぶ。見つからない・形が変わったときは
黙って別の値を出さず、`assert` で止める。

手元で試すとき(AWS に触らない): 環境変数 `OPENDATA_LOCAL_ROOT` にディレクトリを指定すると、
そこをバケットの代わりに読み書きする(手元実行で使う。README の「秋田県オープンデータ全体の評価レポート」参照)。

出典: 秋田県オープンデータカタログサイト(https://ckan.pref.akita.lg.jp/), CC BY 4.0

In [ ]:
# ── 設定 ──────────────────────────────────────────────────────────────────
import os

# True のあいだは HTML を作って表示するだけで S3 には書かない。
DRY_RUN = os.environ.get("DRY_RUN", "1") != "0"

REGION = "ap-northeast-1"
BUCKET_PREFIX = "jaws-festa-2026"
BRONZE_PREFIX = "akita/bronze/opendata/"
MANIFEST_KEY = BRONZE_PREFIX + "_manifest.json"
OUTPUT_PREFIX = "akita/reports/akita/"

# 手元にも同じものを書く(SageMaker 上ではノートブックの隣)。空にすると書かない
LOCAL_OUT = os.environ.get("AKITA_STORY_OUT", "akita_report")

LOCAL_ROOT = os.environ.get("OPENDATA_LOCAL_ROOT")

TITLE = "数字で見る秋田 ― オープンデータでたどる人と暮らし"

In [ ]:
# ── 読み書き先(S3 か、手元のディレクトリ) ──────────────────────────────────
from pathlib import Path


class S3Store:
    def __init__(self, bucket: str, client):
        self.bucket, self.s3 = bucket, client
        self.label = f"s3://{bucket}/"

    def get(self, key: str) -> bytes:
        return self.s3.get_object(Bucket=self.bucket, Key=key)["Body"].read()

    def put(self, key: str, body: bytes, content_type: str) -> None:
        self.s3.put_object(Bucket=self.bucket, Key=key, Body=body, ContentType=content_type)


class LocalStore:
    def __init__(self, root: str):
        self.root = Path(root).resolve()
        self.label = f"{self.root}/"

    def get(self, key: str) -> bytes:
        return (self.root / key).read_bytes()

    def put(self, key: str, body: bytes, content_type: str) -> None:
        p = self.root / key
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_bytes(body)


if LOCAL_ROOT:
    store = LocalStore(LOCAL_ROOT)
else:
    import boto3

    session = boto3.Session(profile_name=os.environ.get("AWS_PROFILE"), region_name=REGION)
    account_id = session.client("sts").get_caller_identity()["Account"]
    store = S3Store(f"{BUCKET_PREFIX}-{account_id}-{REGION}-an", session.client("s3"))
print("読み書き先:", store.label, "/ DRY_RUN =", DRY_RUN)

In [ ]:
# ── bronze から使うファイルを決め打ちで読む ─────────────────────────────────────
import csv
import io
import json
import math
import re
import unicodedata
from collections import Counter, defaultdict

manifest = json.loads(store.get(MANIFEST_KEY))
ds_meta = manifest["datasets"]
used: dict[str, list[str]] = {}  # 出典に載せるデータセット ID → 何に使ったか


def load(dataset: str, name: str, purpose: str) -> list[list[str]]:
    """データセット ID とリソース名(正規表現)で1本だけ選んで、CSV の行を返す。"""
    hits = [f for f in manifest["files"] if f["dataset"] == dataset
            and re.search(name, unicodedata.normalize("NFKC", f["resource_name"] or ""))]
    assert len(hits) == 1, f"{dataset} / {name}: {len(hits)} 本見つかった({[f['resource_name'] for f in hits]})"
    if purpose not in used.setdefault(dataset, []):
        used[dataset].append(purpose)
    text = store.get(BRONZE_PREFIX + hits[0]["file"]).decode("utf-8")
    return list(csv.reader(io.StringIO(text)))


def num(s: str) -> float | None:
    """'1,234' '△ 5' '-' などを数値に。読めなければ None。"""
    t = unicodedata.normalize("NFKC", s).strip().replace(",", "")
    if t[:1] in ("△", "▲"):
        t = "-" + t[1:].strip()
    try:
        return float(t)
    except ValueError:
        return None


def table(rows: list[list[str]], header_row: int = 0) -> list[dict[str, str]]:
    head = [unicodedata.normalize("NFKC", h).strip() for h in rows[header_row]]
    return [dict(zip(head, r)) for r in rows[header_row + 1:] if any(c.strip() for c in r)]

## 1. データを取り出す

| 話題 | データセット | 使う列 |
| --- | --- | --- |
| 人口の推移・出生と死亡 | 年齢別人口流動調査(年報) | 年次推移 1980〜2025 年 |
| 人口ピラミッド | 年齢別人口流動調査(年齢各歳、男女別) | 1980 年と 2025 年の各歳・男女 |
| 市町村ごとの変化 | 年齢別人口流動調査(年報時系列) | 主要人口統計(総人口・65 歳以上の割合) |
| 県外へ移る理由 | 人口移動理由実態調査 | 県外転出・年齢階級別の理由 |
| クマの出没 | クマダス | 目撃日時・獣種・緯度経度 |
| 発電設備 | 風力・水力・地熱・バイオマス・メガソーラーの導入状況 | 各表の合計出力 |
| 落とし物 | 遺失物・拾得物件・返還の取扱状況 | 件数・現金・物品の種類 |
| 何歳で県外へ移るか | 年齢別人口流動調査(年報時系列) | 年齢各歳の県外転入・県外転出 |
| 何月に人が動くか | 年齢別人口流動調査(月報) | 月ごとの社会増減(1980 年〜) |
| 県内ではどこへ移るか | 年齢別人口流動調査(年報時系列) | 25 市町村間の移動 |
| クマが出る時間帯 | クマダス | 目撃日時の時刻 |
| 発電所ができた年 | 各発電の導入状況 | 運転開始(和暦)と出力 |
| 暮らす外国人 | 在住外国人数 | 国籍別(2016〜2021 年末) |
| 水力発電の季節 | 県営水力発電所発電電力量 | 15 発電所の1日ごとの発電量(2017〜2021 年度) |
| 博物館・図書館・公民館 | 社会教育施設一覧(生涯学習・社会教育関係 / 文化・芸術関係) | 施設分類・緯度・経度 |
| 学校・保育施設の場所と数 | 小・中・高・義務教育学校一覧、幼稚園・保育所・認定こども園一覧、確認大学等一覧 | 住所、施設類型 |
| 地震の想定震度 | 地震被害想定データ(被害想定結果)その1〜3 | 27 想定 × 250m メッシュの計測震度・人口・死者数(冬18時) |

学校・保育施設の一覧には緯度・経度が無いので、国土交通省の「大字・町丁目レベル位置参照情報」(大字・町丁目ごとの代表点)と
住所を手元で照らし合わせて位置を決める(外部の住所検索 API は使わない)。照らし合わせられなかった施設は地図に載せず、数を本文に書く。

地震被害想定は 1本 120MB 前後 × 27 本(約 3.4GB)あり、`./scripts/fetch_akita_opendata.py`(50MB 超を飛ばす)では取得しない。
`./scripts/fetch_akita_opendata.py --max-mb 0` で取ったときだけ地図を描き、無ければこの話題を飛ばす。

市町村の比較は 2010 年から。2005〜2006 年の合併が終わって 25 市町村の形になってからでないと、同じ範囲を比べられないため。

In [ ]:
# (1) 人口の推移、出生と死亡(年報の年次推移)
nenji = table(load("050008_toukeika_036", r"年次推移", "人口の推移・出生と死亡"))
pop = {int(r["年月日"][:4]): num(r["総人口(人)"]) for r in nenji}
births = {int(r["年月日"][:4]): num(r["出生者数・合計(人)"]) for r in nenji if num(r["出生者数・合計(人)"])}
deaths = {int(r["年月日"][:4]): num(r["死亡者数・合計(人)"]) for r in nenji if num(r["死亡者数・合計(人)"])}
social = {int(r["年月日"][:4]): num(r["社会増減数・合計(人)"]) for r in nenji if num(r["社会増減数・合計(人)"]) is not None}
y0, y1 = min(pop), max(pop)
cross = next(y for y in sorted(births) if deaths[y] > births[y])  # 亡くなる人が生まれる人を初めて上回った年
print(f"人口 {y0}: {pop[y0]:,.0f} → {y1}: {pop[y1]:,.0f}({pop[y1] / pop[y0] - 1:+.1%})")
print(f"出生 {min(births)}: {births[min(births)]:,.0f} → {y1}: {births[y1]:,.0f} / 死亡 {y1}: {deaths[y1]:,.0f} / 逆転 {cross} 年")

# (2) 人口ピラミッド(年齢各歳。県全体の行だけ)
AGE = re.compile(r"^(\d+)歳(以上)?$")


def pyramid(name: str, year: int) -> dict[str, list[float]]:
    rows = load("050008_toukeika_032", name, "人口ピラミッド")
    head = [unicodedata.normalize("NFKC", h) for h in rows[0]]
    ages = [(j, int(AGE.match(h).group(1))) for j, h in enumerate(head) if AGE.match(h)]
    assert [a for _, a in ages] == list(range(101)), "0〜100 歳の列がそろっていない"
    out = {}
    for r in rows[1:]:
        if r[head.index("市町村コード")] == "5000" and r[head.index("年月日")].startswith(str(year)):
            sex = unicodedata.normalize("NFKC", r[head.index("男女別")])[:1]  # 男 / 女 / 男(女計)
            if sex in ("男", "女") and "計" not in r[head.index("男女別")]:
                out[sex] = [num(r[j]) or 0 for j, _ in ages]
    assert set(out) == {"男", "女"}, out.keys()
    return out


pyr_old = pyramid(r"1980\(昭和55\)年", 1980)
pyr_new = pyramid(r"2020\(令和2\)年", 2025)
peak = lambda p: max(range(101), key=lambda a: p["男"][a] + p["女"][a])  # noqa: E731
print(f"ピラミッドの最も多い年齢: 1980 年 {peak(pyr_old)} 歳 / 2025 年 {peak(pyr_new)} 歳")

# (3) 市町村ごとの変化(主要人口統計)
shuyo = table(load("050008_toukeika_033", r"主要人口統計", "市町村ごとの人口の変化と高齢化"))
by_city: dict[str, dict[int, dict]] = defaultdict(dict)
for r in shuyo:
    by_city[r["市町村"]][int(r["年月日"][:4])] = r
CITY_FROM, CITY_TO = 2010, max(int(r["年月日"][:4]) for r in shuyo)
cities = [c for c, v in by_city.items() if c != "秋田県" and CITY_FROM in v and CITY_TO in v]
assert len(cities) == 25, cities
city_change = {c: num(by_city[c][CITY_TO]["総人口(人)"]) / num(by_city[c][CITY_FROM]["総人口(人)"]) - 1 for c in cities}
city_aged = {c: num(by_city[c][CITY_TO]["65歳以上人口割合(%)"]) for c in cities}
pref_aged = {y: num(r["65歳以上人口割合(%)"]) for y, r in by_city["秋田県"].items()}
pref_change = num(by_city["秋田県"][CITY_TO]["総人口(人)"]) / num(by_city["秋田県"][CITY_FROM]["総人口(人)"]) - 1
print(f"{CITY_FROM}→{CITY_TO} 県 {pref_change:+.1%} / 減り方が小さい {max(city_change, key=city_change.get)} "
      f"大きい {min(city_change, key=city_change.get)} / 65 歳以上 県 {pref_aged[CITY_TO]}%")

In [ ]:
# (4) 県外へ移る理由(人口移動理由実態調査、2019 年以降の表)
riyu = table(load("050008_toukeika_039", r"^3転入", "県外へ移る理由"))
REASONS = ["転勤", "転職・就職", "就職(学生)", "学業", "結婚・離婚", "退職・同居", "住宅事情", "その他"]
riyu_year = max(r["年月日"] for r in riyu)
AGE_GROUPS = ["15〜19歳", "20〜24歳", "25〜29歳", "30〜34歳", "35〜39歳", "40〜49歳", "50〜59歳", "60〜64歳", "65歳以上"]
out_by_age = {}
for r in riyu:
    if r["年月日"] == riyu_year and r["転出入区分"] == "県外転出" and r["男女別"] == "男女計":
        a = unicodedata.normalize("NFKC", r["年齢階級"]).replace("~", "〜")
        if a in AGE_GROUPS:
            out_by_age[a] = {k: num(r[f"{k}(人)"]) or 0 for k in REASONS}
assert set(out_by_age) == set(AGE_GROUPS), out_by_age.keys()
out_total = sum(sum(v.values()) for v in out_by_age.values())
young = sum(sum(out_by_age[a].values()) for a in AGE_GROUPS[:3])
print(f"{riyu_year} 県外転出(回答者) 15 歳以上 {out_total:,.0f} 人 / うち 15〜29 歳 {young / out_total:.0%}")

In [ ]:
# (5) クマの出没(クマダス)
from datetime import datetime

kuma = table(load("050008_shizenhogoka_003", r"kumadas", "クマなどの出没"))
LAT, LON = "x(緯度)", "y(経度)"  # このデータは x が緯度、y が経度
bears = []
for r in kuma:
    if r["獣種"] != "ツキノワグマ":
        continue
    try:
        t = datetime.strptime(r["目撃日時"].strip(), "%Y/%m/%d %H:%M")
    except ValueError:
        continue
    bears.append({"t": t, "kind": r["情報種別"], "city": r["市町村"], "lat": num(r[LAT]), "lon": num(r[LON])})
kuma_first, kuma_last = min(b["t"] for b in bears), max(b["t"] for b in bears)
by_month: dict[int, Counter] = defaultdict(Counter)
for b in bears:
    by_month[b["t"].year][b["t"].month] += 1
bear_year = Counter(b["t"].year for b in bears)
injury_year = Counter(b["t"].year for b in bears if b["kind"] == "人身被害")
top_year = max(bear_year, key=bear_year.get)
print(f"ツキノワグマ {len(bears):,} 件({kuma_first:%Y-%m-%d}〜{kuma_last:%Y-%m-%d})/ 年ごと {dict(sorted(bear_year.items()))}")
print(f"人身被害 {dict(sorted(injury_year.items()))}")

# (6) 発電設備の出力(各表の「合計」の行)。調べた日が表ごとに違う
ENERGY = [
    ("陸上風力", "050008_shigen-ene_003", r"陸上風力"),
    ("洋上風力", "050008_shigen-ene_003", r"洋上風力"),
    ("水力", "050008_shigen-ene_005", r"水力"),
    ("メガソーラー", "050008_shigen-ene_004", r"メガソーラー"),
    ("地熱", "050008_shigen-ene_002", r"地熱"),
    ("バイオマス", "050008_shigen-ene_001", r"バイオマス"),
]
energy = []
for label, ds, name in ENERGY:
    rows = load(ds, name, "発電設備の出力")
    total = next(r for r in rows if r and r[0].strip() == "合計")  # 表が2回載っているものは最初の合計
    kw = [num(c) for c in total if num(c) is not None][-1]  # 合計の行の一番右の数値 = 総出力(kW)
    asof = re.search(r"(\d{4})年(\d{1,2})月", unicodedata.normalize("NFKC", rows[0][0]))
    energy.append({"label": label, "kw": kw, "asof": f"{asof.group(1)}年{int(asof.group(2))}月"})
for e in energy:
    print(f"  {e['label']:<8} {e['kw'] / 1000:>8,.1f} MW({e['asof']}現在)")

# (7) 落とし物(警察本部)
def lost_table(name: str) -> tuple[list[str], dict[int, list[float]]]:
    rows = load("050008_k02005", name, "落とし物")
    head = [unicodedata.normalize("NFKC", c).strip() for c in rows[2]]
    data = {int(r[0]): [num(c) or 0 for c in r[1:]] for r in rows[3:] if r and r[0].strip().isdigit()}
    return head, data


found_head, found = lost_table(r"^拾得物件取扱")
returned_head, returned = lost_table(r"遺失者返還")
lost_year = max(found)
# 1列目以降: 件数, 現金(千円), 物品(点), 物品の内訳…
found_items = {h: v for h, v in zip(found_head[4:], found[lost_year][3:]) if h}
print(f"{lost_year} 年 拾得 {found[lost_year][0]:,.0f} 件 / 現金 {found[lost_year][1] / 10:,.0f} 万円 / "
      f"返還された現金 {returned[lost_year][1] / 10:,.0f} 万円")
print("  物品の内訳:", found_items)

In [ ]:
# (8) 何歳で県外へ移るか(年齢各歳の人口動態。県全体、最新の1年)
kakusai = table(load("050008_toukeika_033", r"^2県・県内25市町村の年齢各歳", "何歳で県外へ移るか"))
kakusai_year = max(r["年月日"] for r in kakusai)
age_net = {}
for r in kakusai:
    a = r["年齢・年齢階級(歳)"].strip()
    if r["年月日"] == kakusai_year and r["県・市町村"] == "秋田県" and a.isdigit() and int(a) <= 90:
        age_net[int(a)] = sum(num(r[f"県外転入・{x}(人)"]) or 0 for x in "男女") - \
            sum(num(r[f"県外転出・{x}(人)"]) or 0 for x in "男女")
assert sorted(age_net) == list(range(91)), sorted(age_net)[:5]
age_net_total = sum(age_net.values())
age_worst = [min(range(15, 20), key=age_net.get), min(range(20, 30), key=age_net.get)]
print(f"{kakusai_year} 県外との出入り {age_net_total:+,.0f} 人 / 出ていく人が多い年齢 {age_worst}"
      f"({', '.join(f'{age_net[a]:+,.0f}' for a in age_worst)})")

# (9) 何月に人が動くか(月報の月次推移)。「各月1日現在」の行の転入・転出は、その前の月の1か月分
getsu = table(load("050008_toukeika_020", r"^4月次データ推移", "何月に人が動くか"))
social_month: dict[tuple[int, int], float] = {}
for r in getsu:
    m_ = re.match(r"(\d{4})年(\d{1,2})月", r["推計年月日"])
    v = num(r["社会増減数(人)"])
    if m_ and v is not None:
        y, mo = int(m_.group(1)), int(m_.group(2))
        y, mo = (y, mo - 1) if mo > 1 else (y - 1, 12)
        social_month[(y, mo)] = v
sm_years = sorted({y for y, _ in social_month})
recent = [y for y in sm_years if y >= sm_years[-1] - 10 and all((y, m) in social_month for m in range(1, 13))]
avg_month = {m: sum(social_month[(y, m)] for y in recent) / len(recent) for m in range(1, 13)}
print(f"月ごとの社会増減 {sm_years[0]}〜{sm_years[-1]} / 直近 {len(recent)} 年の平均 3月 {avg_month[3]:+,.0f} 4月 {avg_month[4]:+,.0f}")

# (10) 県内ではどこへ移るか(25市町村間の移動、最新の1年)。行が移動後、列が移動前
idou = table(load("050008_toukeika_033", r"^5県内25市町村間の移動", "県内ではどこへ移るか"))
idou_year = max(r["年月日"] for r in idou)
flow = {}
for r in idou:
    if r["年月日"] == idou_year and r["移動後の市町村"] in cities:
        flow[r["移動後の市町村"]] = {c: num(r.get(f"【移動前】{c}(人)", "")) or 0 for c in cities}
assert set(flow) == set(cities), set(cities) - set(flow)
akita_net = {c: flow["秋田市"][c] - flow[c]["秋田市"] for c in cities if c != "秋田市"}
akita_gain = sum(akita_net.values())
print(f"{idou_year} 秋田市の県内での転入超過 {akita_gain:+,.0f} 人 / 秋田市が受け入れ超過の市町村 "
      f"{sum(1 for v in akita_net.values() if v > 0)} / {len(akita_net)}")

# (11) クマが出る時間帯(全期間、時刻 × 月)
hour_month = Counter((b["t"].hour, b["t"].month) for b in bears)
by_hour = Counter(b["t"].hour for b in bears)
dawn = sum(by_hour[h] for h in range(5, 9)) / len(bears)
dusk = sum(by_hour[h] for h in range(16, 20)) / len(bears)
print(f"時間帯 5〜8時 {dawn:.0%} / 16〜19時 {dusk:.0%}")

# (12) 発電所ができた年(各表の行。運転開始は和暦「H25. 7」「R4.12」など)
ERA = {"M": 1867, "T": 1911, "S": 1925, "H": 1988, "R": 2018}
ERA_RE = re.compile(r"^([MTSHR])\s*(\d+|元)\s*\.")
plants = []
for label, ds, name in ENERGY:
    rows = load(ds, name, "発電設備の出力")
    head = next(i for i, r in enumerate(rows) if r and unicodedata.normalize("NFKC", r[0]).strip().upper() == "NO.")
    kw_cols = [j for j, h in enumerate(rows[head]) if "出力" in h]
    for r in rows[head + 2:]:
        if r and r[0].strip() == "合計":
            break  # 洋上風力の表は2回載っているので、最初の合計で止める
        m_ = ERA_RE.match(unicodedata.normalize("NFKC", r[1] if len(r) > 1 else "").strip())
        if not m_:
            continue
        kw = [num(r[j]) for j in kw_cols if j < len(r) and num(r[j])]
        if not kw or kw[-1] <= 0:  # 負の値はリプレースで撤去された設備
            continue
        plants.append({"type": label, "year": ERA[m_.group(1)] + (1 if m_.group(2) == "元" else int(m_.group(2))),
                       "kw": kw[-1], "name": r[3].strip() if len(r) > 3 else ""})
oldest_plant = min(plants, key=lambda x: x["year"])
by_type_first = {t: min(x["year"] for x in plants if x["type"] == t) for t, _, _ in ENERGY}
print(f"発電所 {len(plants)} か所 / 最も古い {oldest_plant['name']}({oldest_plant['year']}年・{oldest_plant['type']})")
print("  種類ごとの最初の年:", by_type_first)

# (13) 暮らす外国人(国籍別、各年12月末)
gaikoku_rows = load("050008_kokusaika_001", r"国籍別", "暮らす外国人")
ghead = [unicodedata.normalize("NFKC", c).strip() for c in gaikoku_rows[2]]
G_NEW, G_OLD = ghead[2], ghead[-1]  # 「2021末」と「2016末」
gaikoku = {r[1].strip(): (num(r[-1]) or 0, num(r[2]) or 0) for r in gaikoku_rows[3:]
           if len(r) > 2 and r[0].strip().isdigit()}
gaikoku_total = next(num(r[2]) for r in gaikoku_rows if r and re.sub(r"\s", "", r[0]) == "合計")
fastest = max((k for k, (a, b) in gaikoku.items() if a >= 20), key=lambda k: gaikoku[k][1] / gaikoku[k][0])
print(f"外国人 {G_OLD}→{G_NEW} 合計 {gaikoku_total:,.0f} 人 / 伸びが大きい {fastest} {gaikoku[fastest]}")

In [ ]:
# (14) 学校・保育施設(一覧には緯度・経度が無いので、住所を大字・町丁目の代表点に置く)
import urllib.request
import zipfile

ISJ_URL = "https://nlftp.mlit.go.jp/isj/dls/data/19.0b/05000-19.0b.zip"  # 秋田県・大字町丁目レベル(19.0b)
ISJ_FILE = "05000-19.0b/05_2025.csv"
ISJ_SOURCE = "大字・町丁目レベル位置参照情報 国土交通省(19.0b、2025年整備、05_2025.csv)"


def fetch_isj() -> list[dict]:
    try:
        req = urllib.request.Request(ISJ_URL, headers={"User-Agent": "jaws-festa-2026-akita-story"})
        with urllib.request.urlopen(req, timeout=30) as r:
            z = zipfile.ZipFile(io.BytesIO(r.read()))
        rows = list(csv.DictReader(io.StringIO(z.read(ISJ_FILE).decode("cp932"))))
    except Exception as ex:  # ネットワークに出られない環境では、地図なしで市町村ごとの集計だけ出す
        print(f"位置参照情報を取れなかったので、学校の地図は描かない: {ex}")
        return []
    print(f"位置参照情報 {len(rows):,} 件")
    return rows


isj = fetch_isj()
KANSUJI = dict(zip("一二三四五六七八九", range(1, 10)))


def norm_addr(a: str) -> str:
    # 全角・空白をそろえ、「三丁目」を「3丁目」にする
    a = re.sub(r"\s", "", unicodedata.normalize("NFKC", a))

    def chome(m):
        t = m.group(1)
        if "十" in t:
            x, _, y = t.partition("十")
            return f"{(KANSUJI.get(x, 1) if x else 1) * 10 + KANSUJI.get(y, 0)}丁目"
        return f"{KANSUJI[t]}丁目"
    return re.sub(r"([一二三四五六七八九十]+)丁目", chome, a)


town_pts: dict[str, list[tuple[str, float, float]]] = defaultdict(list)
for r in isj:
    city = re.sub(r"^.+?郡", "", r["市区町村名"])  # 「山本郡藤里町」→「藤里町」
    town_pts[city].append((norm_addr(r["大字町丁目名"]), float(r["経度"]), float(r["緯度"])))
for v in town_pts.values():
    v.sort(key=lambda t: -len(t[0]))  # 長い名前から照らし合わせる(「旭川新藤田西町」を「旭川」より先に)


def city_of(addr: str) -> str | None:
    a = re.sub(r"^(秋田県)?(.+?郡)?", "", norm_addr(addr))
    return next((c for c in cities if a.startswith(c)), None)


def locate(addr: str, city: str | None = None) -> tuple[float, float] | None:
    a = re.sub(r"^(秋田県)?(.+?郡)?", "", norm_addr(addr))
    city = city or city_of(addr)
    if not city:
        return None
    a = a[len(city):] if a.startswith(city) else a
    a = re.sub(r"^(大字|字)", "", a)
    hit = next((t for t in town_pts.get(city, []) if a.startswith(t[0])), None)
    return (hit[1], hit[2]) if hit else None


def rows_of(dataset: str, name: str, purpose: str) -> list[dict]:
    return table(load(dataset, name, purpose))


schools = []  # {"kind", "name", "city", "pos"}
for kind, ds, name in [("小学校", "050008_joho_010", "小学校"), ("中学校", "050008_joho_016", "中学校"),
                       ("義務教育学校", "050008_kyouiku_004", "義務教育学校"), ("高校", "050008_joho_017", "高等学校")]:
    seen = set()
    for r in rows_of(ds, name, "学校・保育施設の場所と数"):
        if r["学校名"] in seen:  # 高校は全日制・定時制などで同じ学校が2行ある
            continue
        seen.add(r["学校名"])
        schools.append({"kind": kind, "name": r["学校名"], "city": city_of(r["住所"]), "pos": locate(r["住所"])})
for r in rows_of("050008_koutoukyouiku_002", "確認大学等", "学校・保育施設の場所と数"):
    schools.append({"kind": "大学・専門学校", "name": r["確認大学等の名称"], "city": city_of(r["確認大学等の所在地"]),
                    "pos": locate(r["確認大学等の所在地"])})


def nursery(name: str) -> list[dict]:
    out = []
    for r in rows_of("050008_kodomo_001" if "8" in name else "050008_youho_001", name, "学校・保育施設の場所と数"):
        t = re.sub(r"^\d+-", "", r["施設類型"])
        t = "認定こども園" if t.startswith("認定こども園") else ("幼稚園" if t.startswith("幼稚園") else t)
        out.append({"kind": "保育施設", "type": t, "name": re.sub(r"\s", "", unicodedata.normalize("NFKC", r["施設名称"])),
                    "city": r["市町村名"].strip(), "pos": locate(r["所在地"], r["市町村名"].strip())})
    return out


nursery_new, nursery_old = nursery(r"令和8年"), nursery(r"令和7年")
facilities = schools + nursery_new
assert all(f["city"] in cities for f in facilities), {f["city"] for f in facilities} - set(cities)
located = [f for f in facilities if f["pos"]]
# 同じ大字・町丁目にある施設は代表点が同じになって重なるので、2つ目からは代表点のまわりに少しずつずらして置く
_dup: Counter = Counter()
for f in located:
    k = _dup[f["pos"]]
    _dup[f["pos"]] += 1
    if k:
        ang, rad = k * 2.4, 0.0045 * math.sqrt(k)  # 黄金角でらせん状に
        f["pos"] = (f["pos"][0] + rad * math.cos(ang) / math.cos(math.radians(f["pos"][1])), f["pos"][1] + rad * math.sin(ang))
print(f"学校・保育施設 {len(facilities)} か所 / 位置が決まった {len(located)}(決まらない {len(facilities) - len(located)})")
print("  ", dict(Counter(f["kind"] for f in facilities)))

# 子どもの数(年齢各歳、最新の年、男女計)
kids_rows = load("050008_toukeika_032", r"2020\(令和2\)年", "学校・保育施設の場所と数")
khead = [unicodedata.normalize("NFKC", h) for h in kids_rows[0]]
kids_year = max(r[khead.index("年月日")] for r in kids_rows[1:])
age_col = {int(m.group(1)): j for j, h in enumerate(khead) if (m := AGE.match(h))}
kids: dict[str, dict[str, float]] = {}
for r in kids_rows[1:]:
    if r[khead.index("年月日")] == kids_year and "男女計" in r[khead.index("男女別")]:
        ages = lambda a, b: sum(num(r[age_col[x]]) or 0 for x in range(a, b + 1))  # noqa: E731
        kids[r[khead.index("市町村")].strip()] = {"保育施設": ages(0, 5), "小学校": ages(6, 11), "中学校": ages(12, 14),
                                                  "高校": ages(15, 17)}
STAGES = [("保育施設", "0〜5歳"), ("小学校", "6〜11歳"), ("中学校", "12〜14歳"), ("高校", "15〜17歳")]


def count_of(city: str | None, stage: str) -> int:
    kinds = {"小学校": ("小学校", "義務教育学校"), "中学校": ("中学校", "義務教育学校")}.get(stage, (stage,))
    return sum(1 for f in facilities if f["kind"] in kinds and (city is None or f["city"] == city))


kids_per = {c: {st: (kids[c][st] / count_of(c, st) if count_of(c, st) else None) for st, _ in STAGES} for c in cities}
pref_kids_per = {st: kids["秋田県"][st] / count_of(None, st) for st, _ in STAGES}
no_high = sorted(c for c in cities if not count_of(c, "高校"))
print(f"{kids_year} 県全体 子ども何人に1か所: " + ", ".join(f"{st} {v:,.0f}" for st, v in pref_kids_per.items()))
print("高校の無い市町村:", no_high)

# 保育施設の1年の変化(名前で突き合わせる)
old_by, new_by = {f["name"]: f for f in nursery_old}, {f["name"]: f for f in nursery_new}
closed = sorted(set(old_by) - set(new_by))
opened = sorted(set(new_by) - set(old_by))
type_old, type_new = Counter(f["type"] for f in nursery_old), Counter(f["type"] for f in nursery_new)
print(f"保育施設 R7 {len(nursery_old)} → R8 {len(nursery_new)} / 名前が消えた {len(closed)} 新しい名前 {len(opened)}")
print("  類型", dict(type_old), "→", dict(type_new))

In [ ]:
# (15) 水力発電の季節(県営水力 15 発電所の合計。年度ごと = 4月1日から)
from datetime import date

hydro: dict[int, dict[int, float]] = {}  # 年度 → {4月1日からの日数: MWh}
for f in [f for f in manifest["files"] if f["dataset"] == "050008_koueikigyou_001"]:
    rows = load(f["dataset"], re.escape(unicodedata.normalize("NFKC", f["resource_name"])), "水力発電の季節")
    plants_cols = [j for j, h in enumerate(rows[0]) if "[MWh]" in h]
    for r in rows[1:]:
        m_ = re.match(r"(\d{4})-(\d{2})-(\d{2})", r[0].strip()) if r else None
        if not m_:
            continue
        d = date(*map(int, m_.groups()))
        fy = d.year if d.month >= 4 else d.year - 1
        day = (d - date(fy, 4, 1)).days
        if day < 365:  # うるう年の3月31日は省く(年度どうしを同じ横軸にそろえるため)
            hydro.setdefault(fy, {})[day] = sum(num(r[j]) or 0 for j in plants_cols if j < len(r))
assert all(len(v) == 365 for v in hydro.values()), {k: len(v) for k, v in hydro.items()}
n_hydro_plants = len(plants_cols)


def moving_avg(v: dict[int, float], k: int = 7) -> dict[float, float]:
    xs = sorted(v)
    return {float(x): sum(v[y] for y in xs[max(0, i - k // 2):i + k // 2 + 1]) / len(xs[max(0, i - k // 2):i + k // 2 + 1])
            for i, x in enumerate(xs)}


MONTH_START = [0, 30, 61, 91, 122, 153, 183, 214, 245, 275, 306, 334]  # 4月1日からの日数(4月〜3月)
MONTH_OF = dict(zip(MONTH_START, [4, 5, 6, 7, 8, 9, 10, 11, 12, 1, 2, 3]))
by_month_hydro = Counter()
for v in hydro.values():
    for day, mwh in v.items():
        by_month_hydro[MONTH_OF[max(m for m in MONTH_START if m <= day)]] += mwh / len(hydro)
hydro_year = sum(by_month_hydro.values())
spring_share = (by_month_hydro[4] + by_month_hydro[5]) / hydro_year
hydro_low = min(by_month_hydro, key=by_month_hydro.get)
print(f"県営水力 {n_hydro_plants} 発電所 {min(hydro)}〜{max(hydro)}年度 / 年平均 {hydro_year / 1000:,.0f} GWh / "
      f"4〜5月 {spring_share:.0%} / いちばん少ない月 {hydro_low}月")

# (16) 博物館・図書館・公民館(緯度・経度が入っている)
CULTURE_KIND = {"図書館": "図書館", "登録博物館": "博物館", "博物館相当施設": "博物館", "博物館類似施設": "博物館",
                "文化会館等": "文化会館・ホール"}
culture = []
for name, col in [(r"生涯学習", "施設分類"), (r"文化・芸術", "区分")]:
    for r in rows_of("050008_kyou-shougai_00" + ("1" if "生涯" in name else "2"), name, "博物館・図書館・公民館"):
        lat, lon = num(r["緯度"]), num(r["経度"])
        if lat and lon and 38.85 < lat < 40.55 and 139.65 < lon < 141.05:
            culture.append({"kind": CULTURE_KIND.get(r[col].strip(), "公民館など"), "name": r["名称1"].strip(),
                            "city": r["市町村"].strip(), "pos": (lon, lat)})
culture_kinds = Counter(c["kind"] for c in culture)
lib_per = {c: sum(1 for x in culture if x["kind"] == "図書館" and x["city"] == c) for c in cities}
print(f"社会教育施設 {len(culture)} か所:", dict(culture_kinds), f"/ 図書館の無い市町村 {sum(1 for v in lib_per.values() if not v)}")

In [ ]:
# (16) 地震の想定震度(地震被害想定、250m メッシュ、27 通りの想定)
# 1本 120MB 前後あるので load() のように全部を文字列にせず、1行ずつ読んで必要な列だけ残す。
# fetch_akita_opendata.py の既定(50MB 超を飛ばす)で取ったときは bronze に無いので、この話題だけ飛ばす
QUAKE_DS = ("050008_bousai_001", "050008_bousai_002", "050008_bousai_003")
QUAKE_PURPOSE = "地震の想定震度"
# 気象庁の震度階級(計測震度の下限)。4以下はまとめる
QUAKE_CLASSES = [("4以下", -9.0), ("5弱", 4.5), ("5強", 5.0), ("6弱", 5.5), ("6強", 6.0), ("7", 6.5)]


def quake_class(v: float) -> int:
    return max(i for i, (_, lo) in enumerate(QUAKE_CLASSES) if v >= lo)


def mesh_sw(code: str) -> tuple[float, float]:
    """250m メッシュ(10桁: 1次 4 + 2次 2 + 3次 2 + 2分の1 1 + 4分の1 1)→ 南西の角の(経度, 緯度)。"""
    lat = int(code[0:2]) / 1.5 + int(code[4]) / 12 + int(code[6]) / 120
    lon = int(code[2:4]) + 100 + int(code[5]) / 8 + int(code[7]) / 80
    for k, div in ((8, 1), (9, 2)):  # 1=南西 2=南東 3=北西 4=北東
        q = int(code[k])
        lat += (q >= 3) / (240 * div)
        lon += (q in (2, 4)) / (160 * div)
    return lon, lat


quake_files = [f for f in manifest["files"] if f["dataset"] in QUAKE_DS]
quake_skipped = [x for x in manifest["not_fetched"] if x["dataset"] in QUAKE_DS]
# bronze に入らなかったもの(公開されているファイルの一部が壊れているなど)。理由は台帳にある
quake_failed = [x for x in manifest["failed"] if x["dataset"] in QUAKE_DS]
quake_mesh: dict[str, list] = {}  # メッシュコード → [27 想定の最大の計測震度, 人口, 市町村, 最大になった想定の番号]
quake_scen: list[dict] = []
if not quake_files:
    assert len(quake_skipped) + len(quake_failed) == 27, f"地震被害想定が bronze にも取得しなかったものにも 27 本無い: {len(quake_skipped)}"
    print(f"地震被害想定は bronze に無い(取得の段階で {len(quake_skipped)} 本を飛ばした)ので、この話題は描かない。"
          "fetch_akita_opendata.py --max-mb 0 で取ると描く")
else:
    SCEN_RE = re.compile(r"^(\d{2})\s*(.+?)\s*(CSV)?$")

    def scen(f: dict) -> tuple[int, str]:
        m_ = SCEN_RE.match(unicodedata.normalize("NFKC", f["resource_name"] or "").strip())
        assert m_, f["resource_name"]
        return int(m_.group(1)), re.sub(r"[\s_]+", "・", m_.group(2))

    by_no = {scen(f)[0]: (f, scen(f)[1]) for f in quake_files}
    quake_missing = {scen(f)[0]: (scen(f)[1], f["problems"]) for f in quake_failed}
    assert sorted([*by_no, *quake_missing]) == list(range(1, 28)), f"想定の番号が 1〜27 でない: {sorted(by_no)}"
    for no, (name, why) in sorted(quake_missing.items()):
        print(f"  {no:02d} {name} は bronze に無いので除く: {why}")
    for ds in QUAKE_DS:
        used.setdefault(ds, []).append(QUAKE_PURPOSE)
    head0 = None
    for no in sorted(by_no):
        f, name = by_no[no]
        rd = csv.reader(io.TextIOWrapper(io.BytesIO(store.get(BRONZE_PREFIX + f["file"])), encoding="utf-8", newline=""))
        head = next(rd)
        assert head0 is None or head == head0, f"{name}: 見出しが 01 と違う"  # 27 本とも同じ 245 列であること
        head0 = head
        iM, iC, iI, iP = (head.index(c) for c in ("メッシュコード", "市町村名", "計測震度", "人口"))
        iD, iZ = head.index("死者数(冬18時)"), head.index("建物全壊(全て)(冬18時)")
        s = {"no": no, "name": name, "pop6": 0.0, "dead": 0.0, "zenkai": 0.0, "max": 0.0, "pop_cls": [0.0] * 6,
             "no_int": 0, "no_int_pop": 0.0}
        for r in rd:
            if not r[iC]:  # 市町村名の無い区画は海・県外
                continue
            if not r[iI]:  # 陸でも計測震度が空の区画がある(地図では塗らない。数だけ出す)
                s["no_int"] += 1
                s["no_int_pop"] += float(r[iP] or 0)
                continue
            v, p = float(r[iI]), float(r[iP] or 0)
            k = quake_class(v)
            s["pop_cls"][k] += p
            s["pop6"] += p if k >= 3 else 0
            s["dead"] += float(r[iD] or 0)
            s["zenkai"] += float(r[iZ] or 0)
            s["max"] = max(s["max"], v)
            m = quake_mesh.get(r[iM])
            if m is None:
                quake_mesh[r[iM]] = [v, p, r[iC], no]
            elif v > m[0]:
                m[0], m[3] = v, no
        quake_scen.append(s)
    quake_pop = sum(m[1] for m in quake_mesh.values())
    quake_pop_cls = [sum(m[1] for m in quake_mesh.values() if quake_class(m[0]) == k) for k in range(6)]
    quake_pop6 = sum(quake_pop_cls[3:])
    quake_worst = max(quake_scen, key=lambda s: s["dead"])
    quake_widest = max(quake_scen, key=lambda s: s["pop6"])
    # 「7」になる区画が、どの想定で 7 になるか
    quake_7_by = Counter(by_no[m[3]][1] for m in quake_mesh.values() if quake_class(m[0]) == 5)
    print(f"地震被害想定 {len(quake_scen)} 想定 / 陸の区画 {len(quake_mesh):,} / 人口 {quake_pop:,.0f}"
          f"(計測震度が空の区画 {quake_scen[0]['no_int']:,}・人口 {quake_scen[0]['no_int_pop']:,.0f} は除いた)")
    print(f"  最大震度ごとの人口: " + ", ".join(f"{QUAKE_CLASSES[k][0]} {quake_pop_cls[k]:,.0f}" for k in range(6)))
    print(f"  6弱以上 {quake_pop6 / quake_pop:.0%} / 死者が最も多い {quake_worst['name']} {quake_worst['dead']:,.0f} 人"
          f" / 6弱以上の人口が最も多い {quake_widest['name']} {quake_widest['pop6']:,.0f} 人")
    print(f"  震度7の区画をもたらす想定: {quake_7_by.most_common(5)}")

## 2. 図を作る

図は SVG を文字列で組み立てる(matplotlib を使わない)。文字はブラウザのフォントで描かれるので、
SageMaker に日本語フォントが無くても化けない。一般の方が読む前提で、図の中に「何が言えるか」を1行書き込む。
CSV 由来の文字は全部 `html.escape` してから埋め込む。

In [ ]:
import html

esc = html.escape
INK, SUB, GRID = "#1f2328", "#59636e", "#e5e7eb"
BLUE, RED, GREEN, ORANGE, PURPLE, GRAY = "#1f6feb", "#d1242f", "#1a7f37", "#bf8700", "#8250df", "#8c959f"


def svg_open(w: int, h: int, label: str) -> str:
    return (f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {w} {h}" width="100%" role="img" '
            f'aria-label="{esc(label)}" style="max-width:{w}px;font-family:inherit"><title>{esc(label)}</title>')


def nice_ticks(lo: float, hi: float, n: int = 5) -> list[float]:
    raw = (hi - lo) / n or 1
    mag = 10 ** math.floor(math.log10(raw))
    step = next(m * mag for m in (1, 2, 2.5, 5, 10) if m * mag >= raw)
    t = [math.floor(lo / step) * step]
    while t[-1] < hi:
        t.append(t[-1] + step)
    return t


def man(v: float) -> str:
    if abs(v) >= 1e4:
        return f"{v / 1e4:,.0f}万" if abs(v) >= 1e5 else f"{v / 1e4:,.1f}万"
    return f"{v:,.0f}"


def line_chart(series: list[tuple[str, str, dict[float, float]]], label: str, w: int = 760, h: int = 300,
               y_fmt=man, x_ticks: list[int] | None = None, x_fmt=str, notes: list[tuple[float, float, str]] = (),
               y_zero: bool = True) -> str:
    """series: (名前, 色, {x: y})。notes: (x, y, 文) を点と一緒に書き込む。"""
    l, r, t, b = 64, 24, 16, 52
    xs = [x for _, _, s in series for x in s]
    ys = [y for _, _, s in series for y in s.values()]
    x0, x1 = min(xs), max(xs)
    yt = nice_ticks(0 if y_zero else min(ys), max(ys))
    X = lambda x: l + (x - x0) / ((x1 - x0) or 1) * (w - l - r)  # noqa: E731
    Y = lambda y: h - b - (y - yt[0]) / ((yt[-1] - yt[0]) or 1) * (h - t - b)  # noqa: E731
    out = [svg_open(w, h, label)]
    for v in yt:
        out.append(f'<line x1="{l}" x2="{w - r}" y1="{Y(v):.1f}" y2="{Y(v):.1f}" stroke="{GRID}"/>'
                   f'<text x="{l - 8}" y="{Y(v) + 4:.1f}" font-size="12" text-anchor="end" fill="{SUB}">{esc(y_fmt(v))}</text>')
    for v in x_ticks or nice_ticks(x0, x1, 8):
        if x0 <= v <= x1:
            out.append(f'<text x="{X(v):.1f}" y="{h - b + 18}" font-size="12" text-anchor="middle" fill="{SUB}">{esc(x_fmt(v))}</text>')
    for i, (name, color, s) in enumerate(series):
        pts = sorted(s.items())
        out.append(f'<polyline fill="none" stroke="{color}" stroke-width="2.5" stroke-linejoin="round" '
                   f'points="{" ".join(f"{X(x):.1f},{Y(y):.1f}" for x, y in pts)}"/>')
        lx = l + i * 150
        out.append(f'<rect x="{lx}" y="{h - 18}" width="14" height="4" fill="{color}"/>'
                   f'<text x="{lx + 20}" y="{h - 12}" font-size="12" fill="{INK}">{esc(name)}</text>')
    for x, y, text in notes:
        anchor = "end" if X(x) > w * 0.6 else "start"
        dx = -8 if anchor == "end" else 8
        out.append(f'<circle cx="{X(x):.1f}" cy="{Y(y):.1f}" r="4.5" fill="#fff" stroke="{INK}" stroke-width="1.8"/>'
                   f'<text x="{X(x) + dx:.1f}" y="{Y(y) - 9:.1f}" font-size="13" font-weight="600" '
                   f'text-anchor="{anchor}" fill="{INK}">{esc(text)}</text>')
    out.append("</svg>")
    return "".join(out)


def hbar(items: list[tuple[str, float]], label: str, w: int = 760, fmt=man, color: str = BLUE,
         highlight: dict[str, str] | None = None, l: int = 120, signed: bool = False) -> str:
    """横棒。signed=True なら 0 を中心にして負の値を左へ伸ばす。"""
    bh = 22
    h = bh * len(items) + 12
    r = 80
    lo = min(0, min(v for _, v in items)) if signed else 0
    hi = max(0, max(v for _, v in items))
    X = lambda v: l + (v - lo) / ((hi - lo) or 1) * (w - l - r)  # noqa: E731
    out = [svg_open(w, h, label)]
    for i, (name, v) in enumerate(items):
        y = 6 + i * bh
        c = (highlight or {}).get(name, color)
        x_a, x_b = sorted((X(0), X(v)))
        out.append(f'<text x="{l - 8}" y="{y + 15}" font-size="12.5" text-anchor="end" fill="{INK}">{esc(name)}</text>'
                   f'<rect x="{x_a:.1f}" y="{y + 3}" width="{max(x_b - x_a, 1):.1f}" height="{bh - 6}" fill="{c}" rx="2"/>')
        tx, anchor = (X(0) + 6, "start") if (signed and v < 0) else (x_b + 4, "start")  # 負の値も 0 の右に書く(左は市町村名とぶつかる)
        out.append(f'<text x="{tx:.1f}" y="{y + 15}" font-size="12" text-anchor="{anchor}" fill="{INK}">{esc(fmt(v))}</text>')
    if signed:
        out.append(f'<line x1="{X(0):.1f}" x2="{X(0):.1f}" y1="4" y2="{h - 4}" stroke="{SUB}"/>')
    out.append("</svg>")
    return "".join(out)

In [ ]:
def pyramid_chart(old: dict, new: dict, old_label: str, new_label: str, w: int = 760) -> str:
    """新しい年を塗りつぶし、古い年を線で重ねる。1 歳 = 1 本。"""
    bh, top, mid_gap = 3.2, 24, 44
    h = int(top + bh * 101 + 56)
    cx = w / 2
    m = max(max(p[s]) for p in (old, new) for s in ("男", "女"))
    half = cx - mid_gap / 2 - 30
    sx = lambda v: v / m * half  # noqa: E731
    Y = lambda a: top + (100 - a) * bh  # noqa: E731
    out = [svg_open(w, h, f"人口ピラミッド {old_label} と {new_label}")]
    for a in range(0, 101, 10):
        out.append(f'<text x="{cx}" y="{Y(a) + bh:.1f}" font-size="11" text-anchor="middle" fill="{SUB}">{a}歳</text>')
    for sex, sign, color in (("男", -1, BLUE), ("女", 1, RED)):
        base = cx + sign * mid_gap / 2
        for a in range(101):
            v = sx(new[sex][a])
            x = base - v if sign < 0 else base
            out.append(f'<rect x="{x:.1f}" y="{Y(a):.1f}" width="{v:.1f}" height="{bh - 0.4:.1f}" fill="{color}" fill-opacity="0.55"/>')
        pts = []
        for a in range(101):
            x = base + sign * sx(old[sex][a])
            pts += [f"{x:.1f},{Y(a) + bh:.1f}", f"{x:.1f},{Y(a):.1f}"]
        out.append(f'<polyline fill="none" stroke="{INK}" stroke-width="1.4" points="{" ".join(pts)}"/>')
        out.append(f'<text x="{base + sign * (half * 0.9):.1f}" y="{top + 6}" font-size="14" font-weight="600" '
                   f'text-anchor="{"start" if sign > 0 else "end"}" fill="{color}">{sex}性</text>')
    for v in nice_ticks(0, m, 3)[1:]:
        if sx(v) <= half:
            for sign in (-1, 1):
                x = cx + sign * (mid_gap / 2 + sx(v))
                out.append(f'<text x="{x:.1f}" y="{top + bh * 101 + 16:.1f}" font-size="11" text-anchor="middle" fill="{SUB}">{v:,.0f}人</text>')
    ly = h - 14
    out.append(f'<rect x="{cx - 220}" y="{ly - 10}" width="8" height="10" fill="{BLUE}" fill-opacity="0.55"/>'
               f'<rect x="{cx - 212}" y="{ly - 10}" width="8" height="10" fill="{RED}" fill-opacity="0.55"/>'
               f'<text x="{cx - 198}" y="{ly}" font-size="12" fill="{INK}">{esc(new_label)}(塗り)</text>'
               f'<line x1="{cx + 20}" x2="{cx + 40}" y1="{ly - 5}" y2="{ly - 5}" stroke="{INK}" stroke-width="1.6"/>'
               f'<text x="{cx + 46}" y="{ly}" font-size="12" fill="{INK}">{esc(old_label)}(線)</text>')
    out.append("</svg>")
    return "".join(out)


def stacked100(rows: list[tuple[str, dict[str, float]]], keys: list[str], colors: list[str], label: str,
               w: int = 760) -> str:
    bh, l, r, top = 26, 90, 16, 8
    legend_rows = (len(keys) + 3) // 4
    h = top + bh * len(rows) + 14 + 20 * legend_rows
    out = [svg_open(w, h, label)]
    for i, (name, vals) in enumerate(rows):
        y = top + i * bh
        tot = sum(vals.values()) or 1
        x = l
        out.append(f'<text x="{l - 8}" y="{y + 17}" font-size="12.5" text-anchor="end" fill="{INK}">{esc(name)}</text>')
        for k, c in zip(keys, colors):
            ww = vals[k] / tot * (w - l - r)
            out.append(f'<rect x="{x:.1f}" y="{y + 3}" width="{ww:.1f}" height="{bh - 6}" fill="{c}"/>')
            if ww >= 34:
                out.append(f'<text x="{x + ww / 2:.1f}" y="{y + 17}" font-size="11" text-anchor="middle" '
                           f'fill="#fff">{vals[k] / tot * 100:.0f}%</text>')
            x += ww
    for j, (k, c) in enumerate(zip(keys, colors)):
        lx = l + (j % 4) * ((w - l) / 4)
        ly = top + bh * len(rows) + 22 + 20 * (j // 4)
        out.append(f'<rect x="{lx:.1f}" y="{ly - 10}" width="12" height="12" fill="{c}"/>'
                   f'<text x="{lx + 17:.1f}" y="{ly}" font-size="12" fill="{INK}">{esc(k)}</text>')
    out.append("</svg>")
    return "".join(out)


# ── 背景の地図(地理院タイル・淡色地図) ───────────────────────────────────────────
# 地図の画像はノートブックを動かすときに国土地理院から取ってきて、HTML に data URI で埋め込む
# (ページを開いた人のブラウザから外へ読みに行かないように)。取れなければ地図なしで描く。
import base64
import urllib.request

TILE_URL = "https://cyberjapandata.gsi.go.jp/xyz/pale/{z}/{x}/{y}.png"
TILE_Z = 9  # 秋田県がおよそ 510 x 805 px になる(タイル 12 枚)
AKITA_BBOX = (139.65, 38.85, 141.05, 40.55)  # 経度・緯度のおおよその範囲(西, 南, 東, 北)


def merc(lon: float, lat: float, z: int = TILE_Z) -> tuple[float, float]:
    """経度・緯度 → Web メルカトルの画素座標(地理院タイルと同じ)。"""
    n = 256 * 2 ** z
    s = math.sin(math.radians(lat))
    return (lon + 180) / 360 * n, (0.5 - math.log((1 + s) / (1 - s)) / (4 * math.pi)) * n


def fetch_tiles(bbox: tuple[float, float, float, float], z: int = TILE_Z) -> list[tuple[int, int, bytes]]:
    x0, y0 = merc(bbox[0], bbox[3], z)
    x1, y1 = merc(bbox[2], bbox[1], z)
    tiles = []
    try:
        for ty in range(int(y0 // 256), int(y1 // 256) + 1):
            for tx in range(int(x0 // 256), int(x1 // 256) + 1):
                req = urllib.request.Request(TILE_URL.format(z=z, x=tx, y=ty),
                                             headers={"User-Agent": "jaws-festa-2026-akita-story"})
                with urllib.request.urlopen(req, timeout=10) as r:
                    body = r.read()
                assert body[:4] == b"\x89PNG", "PNG でない"
                tiles.append((tx, ty, body))
    except Exception as ex:  # ネットワークに出られない環境(VPC のみ等)では地図なしで続ける
        print(f"地理院タイルを取れなかったので、地図なしで描く: {ex}")
        return []
    print(f"地理院タイル {len(tiles)} 枚({sum(len(b) for _, _, b in tiles) / 1e3:,.0f} KB)")
    return tiles


basemap = fetch_tiles(AKITA_BBOX)
_basemap_defined = False
# 出典と、加工したこと(点と地名を書き足した)を分けて書く(国土地理院コンテンツ利用規約 / 承認申請 Q&A Q1-15)
GSI_TILES_URL = "https://maps.gsi.go.jp/development/ichiran.html"
GSI_SOURCE = "出典: 国土地理院 地理院タイル(淡色地図)"
GSI_EDIT = "地理院タイルにクマの出没地点と地名を追記して掲載"
GSI_EDIT_SCHOOL = "地理院タイルに学校・保育施設の位置を追記して掲載"
GSI_EDIT_CULTURE = "地理院タイルに博物館・図書館・公民館などの位置を追記して掲載"


def dot_map(points: list[tuple], labels: list[tuple[str, float, float]], label: str,
            color: str = RED, max_points: int = 6000, r: float = 1.8, opacity: float = 0.4,
            legend: list[tuple[str, str]] = (), edit: str = "", unit: str = "1点 = 1件") -> str:
    """経度・緯度の点を、地理院タイルの上に打つ。座標は Web メルカトルの画素(ズーム TILE_Z)。"""
    if len(points) > max_points:  # 点が多すぎると HTML が重くなるので一定の間隔で間引く
        step = len(points) / max_points
        points = [points[int(i * step)] for i in range(max_points)]
    x0, y0 = merc(AKITA_BBOX[0], AKITA_BBOX[3])
    x1, y1 = merc(AKITA_BBOX[2], AKITA_BBOX[1])
    w, h = x1 - x0, y1 - y0
    out = [svg_open(round(w), round(h), label).replace(f'viewBox="0 0 {round(w)} {round(h)}"',
                                                       f'viewBox="{x0:.1f} {y0:.1f} {w:.1f} {h:.1f}"'),
           f'<rect x="{x0:.1f}" y="{y0:.1f}" width="{w:.1f}" height="{h:.1f}" fill="#f6f8fa"/>']
    # 地図の画像は最初の図にだけ埋め込み、2枚目からは同じページの中で参照する(HTML を軽くするため)
    global _basemap_defined
    if basemap and not _basemap_defined:
        out.append('<g id="gsi-basemap">' + "".join(
            f'<image x="{tx * 256}" y="{ty * 256}" width="256" height="256" '
            f'href="data:image/png;base64,{base64.b64encode(body).decode()}"/>' for tx, ty, body in basemap) + "</g>")
        _basemap_defined = True
    elif basemap:
        out.append('<use href="#gsi-basemap"/>')
    for pt in points:  # (経度, 緯度) か (経度, 緯度, 色)
        x, y = merc(pt[0], pt[1])
        c = pt[2] if len(pt) > 2 else color
        out.append(f'<circle cx="{x:.1f}" cy="{y:.1f}" r="{r}" fill="{c}" fill-opacity="{opacity}"'
                   + (' stroke="#fff" stroke-width="0.8"/>' if r > 2.5 else "/>"))
    for name, lon, lat in labels:
        x, y = merc(lon, lat)
        out.append(f'<text x="{x:.1f}" y="{y:.1f}" font-size="14" font-weight="700" text-anchor="middle" '
                   f'fill="{INK}" stroke="#fff" stroke-width="3.5" paint-order="stroke">{esc(name)}</text>')
    for k, (name, c) in enumerate(legend):
        out.append(f'<rect x="{x0 + 10:.1f}" y="{y0 + 10 + 20 * k:.1f}" width="190" height="20" fill="#fff" fill-opacity="0.85"/>'
                   f'<circle cx="{x0 + 22:.1f}" cy="{y0 + 20 + 20 * k:.1f}" r="5" fill="{c}" stroke="#fff"/>'
                   f'<text x="{x0 + 34:.1f}" y="{y0 + 24 + 20 * k:.1f}" font-size="13" fill="{INK}">{esc(name)}</text>')
    lines = ([GSI_SOURCE, edit or GSI_EDIT] if basemap else []) + [f"北が上 / {unit}"]
    for k, line in enumerate(reversed(lines)):
        out.append(f'<text x="{x1 - 8:.1f}" y="{y1 - 8 - 15 * k:.1f}" font-size="11" text-anchor="end" fill="{INK}" '
                   f'stroke="#fff" stroke-width="3" paint-order="stroke">{esc(line)}</text>')
    out.append("</svg>")
    return "".join(out)

In [ ]:
def columns_signed(values: dict[int, float], label: str, w: int = 760, h: int = 280, x_step: int = 10,
                   notes: list[int] = (), unit: str = "人") -> str:
    # 0 を中心に、正は上・負は下へ伸ばす縦棒(1 本 = 1 つの x)
    l, r, t, b = 56, 12, 18, 34
    xs = sorted(values)
    lo, hi = min(0, min(values.values())), max(0, max(values.values()))
    yt = nice_ticks(lo, hi, 5)
    Y = lambda v: t + (yt[-1] - v) / ((yt[-1] - yt[0]) or 1) * (h - t - b)  # noqa: E731
    bw = (w - l - r) / len(xs)
    out = [svg_open(w, h, label)]
    for v in yt:
        out.append(f'<line x1="{l}" x2="{w - r}" y1="{Y(v):.1f}" y2="{Y(v):.1f}" stroke="{GRID if v else SUB}"/>'
                   f'<text x="{l - 6}" y="{Y(v) + 4:.1f}" font-size="11" text-anchor="end" fill="{SUB}">{v:+,.0f}</text>')
    for i, x in enumerate(xs):
        v = values[x]
        y_a, y_b = sorted((Y(0), Y(v)))
        out.append(f'<rect x="{l + i * bw + 0.5:.1f}" y="{y_a:.1f}" width="{max(bw - 1, 1):.1f}" height="{max(y_b - y_a, 0.5):.1f}" '
                   f'fill="{BLUE if v >= 0 else RED}"/>')
        if x % x_step == 0:
            out.append(f'<text x="{l + (i + 0.5) * bw:.1f}" y="{h - b + 16}" font-size="11" text-anchor="middle" fill="{SUB}">{x}歳</text>')
    for x in notes:
        i = xs.index(x)
        v = values[x]
        out.append(f'<text x="{l + (i + 1) * bw + 4:.1f}" y="{Y(v) + 4:.1f}" font-size="12.5" font-weight="700" '
                   f'fill="{INK}" stroke="#fff" stroke-width="3" paint-order="stroke">{x}歳 {v:+,.0f}{unit}</text>')
    out.append("</svg>")
    return "".join(out)


def grid_heatmap(rows: list, cols: list, value, label: str, row_fmt=str, col_fmt=str, diverging: bool = False,
                 w: int = 760, cell_h: float = 9, row_every: int = 5, l: int = 52, legend: str = "") -> str:
    # 行 × 列のマス目。value(行, 列) が None のマスは灰色。diverging なら正を青・負を赤
    top = 26
    cw = (w - l - 10) / len(cols)
    h = top + cell_h * len(rows) + 30
    vals = [v for r_ in rows for c in cols if (v := value(r_, c)) is not None]
    m = max(abs(v) for v in vals) or 1
    out = [svg_open(w, h, label)]
    for j, c in enumerate(cols):
        out.append(f'<text x="{l + (j + 0.5) * cw:.1f}" y="{top - 8}" font-size="11" text-anchor="middle" fill="{SUB}">{esc(col_fmt(c))}</text>')
    for i, r_ in enumerate(rows):
        y = top + i * cell_h
        if i % row_every == 0:
            out.append(f'<text x="{l - 6}" y="{y + cell_h - 1:.1f}" font-size="11" text-anchor="end" fill="{SUB}">{esc(row_fmt(r_))}</text>')
        for j, c in enumerate(cols):
            v = value(r_, c)
            x = l + j * cw
            if v is None:
                fill, op = "#eaeef2", 1
            elif diverging:
                fill, op = (BLUE if v >= 0 else RED), min(1, abs(v) / m) ** 0.6
            else:
                fill, op = RED, (v / m) ** 0.6
            out.append(f'<rect x="{x:.1f}" y="{y:.1f}" width="{cw - 0.6:.1f}" height="{cell_h - 0.6:.1f}" fill="{fill}" fill-opacity="{op:.2f}"/>')
    if legend:
        out.append(f'<text x="{l}" y="{h - 8}" font-size="11.5" fill="{SUB}">{esc(legend)}</text>')
    out.append("</svg>")
    return "".join(out)


def bubble_timeline(items: list[dict], kinds: list[tuple[str, str]], label: str, w: int = 760,
                    x0: int = 1890, x1: int = 2030) -> str:
    # 横軸が年、行が種類、円の面積が出力
    l, r, t, rh = 100, 16, 14, 50
    h = t + rh * len(kinds) + 40
    X = lambda x: l + (x - x0) / (x1 - x0) * (w - l - r)  # noqa: E731
    mk = max(x["kw"] for x in items)
    R_ = lambda kw: 2 + 20 * math.sqrt(kw / mk)  # noqa: E731
    out = [svg_open(w, h, label)]
    for v in range(x0, x1 + 1, 10):
        out.append(f'<line x1="{X(v):.1f}" x2="{X(v):.1f}" y1="{t}" y2="{t + rh * len(kinds)}" stroke="{GRID}"/>'
                   + (f'<text x="{X(v):.1f}" y="{t + rh * len(kinds) + 16}" font-size="11" text-anchor="middle" fill="{SUB}">{v}</text>'
                      if v % 20 == 10 else ""))
    for i, (kind, color) in enumerate(kinds):
        y = t + rh * i + rh / 2
        out.append(f'<text x="{l - 10}" y="{y + 4:.1f}" font-size="12.5" text-anchor="end" fill="{INK}">{esc(kind)}</text>')
        for x in sorted((x for x in items if x["type"] == kind), key=lambda x: -x["kw"]):
            out.append(f'<circle cx="{X(x["year"] + 0.5):.1f}" cy="{y:.1f}" r="{R_(x["kw"]):.1f}" fill="{color}" '
                       f'fill-opacity="0.45" stroke="{color}" stroke-width="0.8"/>')
    out.append(f'<text x="{l}" y="{h - 6}" font-size="11.5" fill="{SUB}">1つの円が1か所。円の面積が出力(最大 {mk / 1000:,.0f} MW)</text>')
    out.append("</svg>")
    return "".join(out)


def slope_chart(items: dict[str, tuple[float, float]], left: str, right: str, label: str, w: int = 760,
                h: int = 360, highlight: set[str] = frozenset()) -> str:
    # 2 時点の値を線で結ぶ。名前は右端に、重ならないように縦にずらして書く
    l, r, t, b = 150, 190, 50, 16
    h = max(h, t + b + 16 * len(items))
    # 縦は対数(人数の差が大きく、増え方を比べたいので)
    lo, hi = math.log10(max(1, min(min(v) for v in items.values()))), math.log10(max(max(v) for v in items.values()))
    Y = lambda v: t + (hi - math.log10(max(v, 1))) / ((hi - lo) or 1) * (h - t - b)  # noqa: E731
    out = [svg_open(w, h, label),
           f'<text x="{l}" y="{t - 12}" font-size="13" font-weight="700" text-anchor="middle" fill="{INK}">{esc(left)}</text>'
           f'<text x="{w - r}" y="{t - 12}" font-size="13" font-weight="700" text-anchor="middle" fill="{INK}">{esc(right)}</text>']

    def spread(ys: list[tuple[float, str]]) -> dict[str, float]:
        # 上から順に 15px ずつ空けて置き、下にはみ出したら下から詰め直す
        order = sorted(ys)
        pos = []
        for y, _ in order:
            pos.append(max(y, pos[-1] + 15) if pos else y)
        for i in range(len(pos) - 1, -1, -1):
            pos[i] = min(pos[i], (pos[i + 1] - 15) if i + 1 < len(pos) else h - b)
        return {k: y for (_, k), y in zip(order, pos)}

    ly = spread([(Y(a), k) for k, (a, _) in items.items()])
    ry = spread([(Y(b_), k) for k, (_, b_) in items.items()])
    for k, (a, b_) in items.items():
        c = RED if k in highlight else GRAY
        sw = 2.6 if k in highlight else 1.4
        bold = ' font-weight="700"' if k in highlight else ""
        out.append(f'<line x1="{l}" x2="{w - r}" y1="{Y(a):.1f}" y2="{Y(b_):.1f}" stroke="{c}" stroke-width="{sw}"/>'
                   f'<circle cx="{l}" cy="{Y(a):.1f}" r="3" fill="{c}"/><circle cx="{w - r}" cy="{Y(b_):.1f}" r="3" fill="{c}"/>'
                   f'<text x="{l - 8}" y="{ly[k] + 4:.1f}" font-size="12" text-anchor="end" fill="{INK}">{esc(k)} {a:,.0f}</text>'
                   f'<text x="{w - r + 8}" y="{ry[k] + 4:.1f}" font-size="12" fill="{INK}"{bold}>{b_:,.0f} {esc(k)}</text>')
    out.append("</svg>")
    return "".join(out)

In [ ]:
def ratio_table(rows: list[str], cols: list[tuple[str, str]], value, pref: dict, unit: str = "人") -> str:
    # 市町村 × 段階の表。値が大きい(子どもが多いのに施設が少ない)ほど濃い。施設が無いマスは「なし」
    # 色の濃さは列ごとに決める(高校は学校が少ないので、全体で決めると高校の列だけ濃くなる)
    hi = {k: max(v for c in rows if (v := value(c, k)) is not None) for k, _ in cols}
    head = "".join(f"<th>{esc(k)}<br><small>{esc(a)}</small></th>" for k, a in cols)
    body = []
    for c in rows:
        tds = []
        for k, _ in cols:
            v = value(c, k)
            if v is None:
                tds.append('<td class="none">なし</td>')
            else:
                op = 0.08 + 0.8 * (v / hi[k])
                tds.append(f'<td style="background:rgba(31,111,235,{op:.2f});color:{"#fff" if op > 0.5 else INK}">'
                           f'{v:,.0f}</td>')
        body.append(f"<tr><th>{esc(c)}</th>{''.join(tds)}</tr>")
    foot = "".join(f"<td><b>{pref[k]:,.0f}</b></td>" for k, _ in cols)
    return (f'<table class="ratio"><thead><tr><th></th>{head}</tr></thead><tbody>{"".join(body)}</tbody>'
            f'<tfoot><tr><th>県全体</th>{foot}</tr></tfoot></table>')

In [ ]:
# ── 市町村の形(国土数値情報 行政区域データ)と、塗り分けた地図 ─────────────────────────
# 境界は実行するときに国土交通省から取ってきて、点を間引いて SVG の path にする。
# マウスを乗せたときの数字は SVG の <title>(ブラウザの標準のツールチップ)で出す。JavaScript は使わない。
N03_URL = "https://nlftp.mlit.go.jp/ksj/gml/data/N03/N03-2025/N03-20250101_05_GML.zip"
N03_PAGE = "https://nlftp.mlit.go.jp/ksj/gml/datalist/KsjTmplt-N03-2025.html"
N03_SOURCE = "「国土数値情報(行政区域データ)」(国土交通省、2025年1月1日時点)を加工して作成"


def fetch_n03() -> dict[str, list[list[tuple[float, float]]]]:
    try:
        req = urllib.request.Request(N03_URL, headers={"User-Agent": "jaws-festa-2026-akita-story"})
        with urllib.request.urlopen(req, timeout=60) as r:
            z = zipfile.ZipFile(io.BytesIO(r.read()))
        gj = json.loads(z.read(next(n for n in z.namelist() if n.endswith(".geojson"))))
    except Exception as ex:  # ネットワークに出られない環境では、塗り分けの地図は描かない
        print(f"行政区域データを取れなかったので、市町村の地図は描かない: {ex}")
        return {}
    rings: dict[str, list] = defaultdict(list)
    for f in gj["features"]:
        city = f["properties"]["N03_004"]
        polys = f["geometry"]["coordinates"]
        if f["geometry"]["type"] == "Polygon":
            polys = [polys]
        for poly in polys:
            for ring in poly:
                rings[city].append([merc(x, y) for x, y in ring])
    return rings


def simplify(pts: list[tuple[float, float]], eps: float) -> list[tuple[float, float]]:
    # ダグラス・ポイカー法(画素の単位で eps より細かい凸凹を落とす)
    keep = [False] * len(pts)
    keep[0] = keep[-1] = True
    stack = [(0, len(pts) - 1)]
    while stack:
        a, b = stack.pop()
        (x1, y1), (x2, y2) = pts[a], pts[b]
        dx, dy = x2 - x1, y2 - y1
        n = math.hypot(dx, dy) or 1e-9
        far, idx = 0.0, None
        for i in range(a + 1, b):
            d = abs(dy * (pts[i][0] - x1) - dx * (pts[i][1] - y1)) / n
            if d > far:
                far, idx = d, i
        if idx is not None and far > eps:
            keep[idx] = True
            stack += [(a, idx), (idx, b)]
    return [p_ for p_, k in zip(pts, keep) if k]


def ring_area(pts: list[tuple[float, float]]) -> float:
    return abs(sum(x1 * y2 - x2 * y1 for (x1, y1), (x2, y2) in zip(pts, pts[1:] + pts[:1]))) / 2


city_rings = fetch_n03()
city_paths: dict[str, str] = {}
for city, rings in city_rings.items():
    parts = []
    for ring in rings:
        if ring_area(ring) < 3:  # 3 画素(約 0.15 km²)より小さい島は省く
            continue
        # 閉じた輪は始点と終点が同じなので、始点からいちばん遠い点で2つに分けて間引く
        far = max(range(len(ring)), key=lambda i: (ring[i][0] - ring[0][0]) ** 2 + (ring[i][1] - ring[0][1]) ** 2)
        sp = simplify(ring[:far + 1], 0.5) + simplify(ring[far:], 0.5)[1:]
        if len(sp) >= 4:
            parts.append("M" + "L".join(f"{x:.1f},{y:.1f}" for x, y in sp) + "Z")
    city_paths[city] = "".join(parts)
if city_paths:
    assert set(city_paths) == set(cities), set(cities) ^ set(city_paths)
    print(f"行政区域 {len(city_paths)} 市町村 / path {sum(len(v) for v in city_paths.values()) / 1e3:,.0f} KB")


def choropleth(values: dict[str, float], bins: list[float], colors: list[str], label: str, legend_fmt,
               tooltip: dict[str, str], w: int = 400) -> str:
    # 市町村を値の階級で塗り分ける。bins は階級の境目(小さい順)、colors は len(bins) + 1 色
    if not city_paths:
        return ""
    xs = [x for rs in city_rings.values() for r_ in rs for x, _ in r_]
    ys = [y for rs in city_rings.values() for r_ in rs for _, y in r_]
    x0, x1, y0, y1 = min(xs) - 4, max(xs) + 4, min(ys) - 4, max(ys) + 4
    leg_h = 22 * (len(colors)) + 10
    out = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="{x0:.0f} {y0:.0f} {x1 - x0:.0f} {y1 - y0 + leg_h:.0f}" '
           f'width="100%" role="img" aria-label="{esc(label)}" class="choro" style="max-width:{w}px;font-family:inherit">'
           f'<title>{esc(label)}</title>']
    for city, d in city_paths.items():
        k = sum(1 for b in bins if values[city] >= b)
        out.append(f'<path d="{d}" fill="{colors[k]}" fill-rule="evenodd" stroke="#fff" stroke-width="0.8">'
                   f'<title>{esc(tooltip[city])}</title></path>')
    for k, c in enumerate(colors):
        lo = bins[k - 1] if k else None
        hi = bins[k] if k < len(bins) else None
        ly = y1 + 6 + 22 * k
        out.append(f'<rect x="{x0 + 8:.0f}" y="{ly:.0f}" width="18" height="16" fill="{c}"/>'
                   f'<text x="{x0 + 32:.0f}" y="{ly + 14:.0f}" font-size="16" fill="{INK}">{esc(legend_fmt(lo, hi))}</text>')
    out.append("</svg>")
    return "".join(out)

In [ ]:
# ── 図 ────────────────────────────────────────────────────────────────────
fig = {}
fig["pop"] = line_chart([("総人口", BLUE, pop)], "秋田県の総人口", y_zero=False,
                        x_ticks=list(range(1980, y1 + 1, 5)),
                        notes=[(y0, pop[y0], f"{y0}年 {pop[y0] / 1e4:,.1f}万人"), (y1, pop[y1], f"{y1}年 {pop[y1] / 1e4:,.1f}万人")])
fig["births"] = line_chart([("生まれた人", GREEN, births), ("亡くなった人", GRAY, deaths)], "出生者数と死亡者数",
                           x_ticks=list(range(1980, y1 + 1, 5)),
                           notes=[(cross, deaths[cross], f"{cross}年に逆転"),
                                  (y1, births[y1], f"{y1}年 {births[y1]:,.0f}人"),
                                  (y1, deaths[y1], f"{deaths[y1]:,.0f}人")])
fig["pyramid"] = pyramid_chart(pyr_old, pyr_new, "1980年", "2025年")

change_items = sorted(((c, v * 100) for c, v in city_change.items()), key=lambda x: -x[1])
pop_now_city = {c: num(by_city[c][CITY_TO]["総人口(人)"]) for c in cities}
pop_then_city = {c: num(by_city[c][CITY_FROM]["総人口(人)"]) for c in cities}
city_tip = {c: f"{c}\n人口 {pop_now_city[c]:,.0f}人({CITY_TO}年)\n{CITY_FROM}年 {pop_then_city[c]:,.0f}人から {city_change[c] * 100:+.1f}%\n"
               f"65歳以上 {city_aged[c]:.1f}%" for c in cities}


def man_label(v: float) -> str:
    return f"{v / 1e4:g}万人" if v >= 1e4 else f"{v:,.0f}人"


POP_BINS = [5000, 10000, 30000, 50000, 100000]
fig["pop_map"] = choropleth(
    pop_now_city, POP_BINS, ["#eff6ff", "#c6dbef", "#9ecae1", "#6baed6", "#3182bd", "#08519c"],
    f"市町村の人口({CITY_TO}年)", lambda lo, hi: (f"{man_label(lo)}以上" if hi is None else
                                              f"{man_label(hi)}未満" if lo is None else f"{man_label(lo)}〜{man_label(hi)}"),
    city_tip)
CHANGE_BINS = [-35, -30, -25, -20, -15]
fig["change_map"] = choropleth(
    {c: v * 100 for c, v in city_change.items()}, CHANGE_BINS,
    ["#99000d", "#cb181d", "#ef3b2c", "#fb6a4a", "#fc9272", "#fdd0a2"],
    f"人口の増減({CITY_FROM}年→{CITY_TO}年)", lambda lo, hi: (f"{lo:+.0f}% より小さい減り" if hi is None else
                                                          f"{hi:+.0f}% より大きい減り" if lo is None else f"{lo:+.0f}%〜{hi:+.0f}%"),
    city_tip)
fig["city_change"] = hbar(change_items, f"市町村ごとの人口の増減({CITY_FROM}年→{CITY_TO}年)", signed=True,
                          fmt=lambda v: f"{v:+.0f}%", color=ORANGE, highlight={"秋田市": BLUE})
aged_items = sorted(city_aged.items(), key=lambda x: -x[1])
fig["city_aged"] = hbar(aged_items, f"市町村ごとの65歳以上の割合({CITY_TO}年)", fmt=lambda v: f"{v:.1f}%",
                        color=PURPLE, highlight={"秋田市": BLUE})
fig["aged_trend"] = line_chart([("65歳以上の割合", PURPLE, pref_aged)], "秋田県の65歳以上の割合",
                               y_fmt=lambda v: f"{v:.0f}%", x_ticks=list(range(1980, CITY_TO + 1, 5)),
                               notes=[(1980, pref_aged[1980], f"1980年 {pref_aged[1980]:.1f}%"),
                                      (CITY_TO, pref_aged[CITY_TO], f"{CITY_TO}年 {pref_aged[CITY_TO]:.1f}%")])

REASON_COLORS = [BLUE, GREEN, "#2da44e", ORANGE, RED, PURPLE, "#57606a", "#afb8c1"]
fig["reasons"] = stacked100([(a, out_by_age[a]) for a in AGE_GROUPS], REASONS, REASON_COLORS,
                            f"県外へ移る理由(年齢別、{riyu_year})")

MONTH_COLORS = {2022: "#afb8c1", 2023: ORANGE, 2024: GREEN, 2025: RED, 2026: BLUE}
fig["kuma_month"] = line_chart(
    [(f"{y}年", MONTH_COLORS.get(y, GRAY), {float(m): float(by_month[y][m]) for m in range(1, 13)
                                            if (y, m) >= (kuma_first.year, kuma_first.month)
                                            and (y, m) <= (kuma_last.year, kuma_last.month)})
     for y in sorted(by_month)],
    "ツキノワグマの出没件数(月別)", x_ticks=list(range(1, 13)), x_fmt=lambda v: f"{int(v)}月", y_fmt=lambda v: f"{v:,.0f}")
pts = [(b["lon"], b["lat"]) for b in bears if b["t"].year == top_year and b["lat"] and b["lon"]
       and 139.65 < b["lon"] < 141.05 and 38.85 < b["lat"] < 40.55]
# 地名は、その市町村で記録された出没地点の中央(中央値)に置く。役所の位置ではない
city_pts: dict[str, list[tuple[float, float]]] = defaultdict(list)
for b in bears:
    if b["lat"] and b["lon"] and 139.65 < b["lon"] < 141.05 and 38.85 < b["lat"] < 40.55:
        city_pts[b["city"]].append((b["lon"], b["lat"]))
MAP_LABELS = ["秋田市", "能代市", "大館市", "鹿角市", "北秋田市", "大仙市", "横手市", "湯沢市", "由利本荘市", "仙北市"]
labels = []
for c in MAP_LABELS:
    if len(city_pts[c]) >= 20:
        lons, lats = sorted(p[0] for p in city_pts[c]), sorted(p[1] for p in city_pts[c])
        labels.append((c, lons[len(lons) // 2], lats[len(lats) // 2]))
fig["kuma_map"] = dot_map(pts, labels, f"{top_year}年のツキノワグマの出没地点")
kuma_city = Counter(b["city"] for b in bears if b["t"].year == top_year)

fig["energy"] = hbar([(e["label"], e["kw"] / 1000) for e in energy], "発電設備の出力(MW)",
                     fmt=lambda v: f"{v:,.0f} MW", color=GREEN, highlight={"陸上風力": BLUE, "洋上風力": BLUE})

fig["lost_items"] = hbar(sorted(((k, v) for k, v in found_items.items() if k != "その他の物品"), key=lambda x: -x[1]),
                         f"届けられた落とし物の種類({lost_year}年)", fmt=lambda v: f"{v:,.0f} 点", color=ORANGE, l=150)
fig["age_net"] = columns_signed(age_net, f"年齢別の県外との出入り({kakusai_year[:4]}年)", notes=age_worst)
fig["social_month"] = grid_heatmap(
    sm_years, list(range(1, 13)), lambda y, m: social_month.get((y, m)), "月ごとの社会増減", col_fmt=lambda m: f"{m}月",
    diverging=True, legend="青: 転入が多い月 / 赤: 転出が多い月(色が濃いほど人数が多い)。灰色はデータの無い月")
fig["akita_pull"] = hbar(sorted(akita_net.items(), key=lambda x: -x[1]), "秋田市との県内での出入り", signed=True,
                         fmt=lambda v: f"{v:+,.0f}人", color=BLUE, highlight={c: RED for c, v in akita_net.items() if v < 0})
fig["kuma_hour"] = grid_heatmap(
    list(range(24)), list(range(1, 13)), lambda hr, m: float(hour_month[(hr, m)]), "ツキノワグマの出没の時刻と月",
    row_fmt=lambda hr: f"{hr}時", col_fmt=lambda m: f"{m}月", cell_h=16, row_every=3,
    legend="色が濃いほど件数が多い(全期間の合計)")
ENERGY_COLORS = {"陸上風力": BLUE, "洋上風力": "#0a3069", "水力": "#1a7f37", "メガソーラー": ORANGE, "地熱": RED,
                 "バイオマス": "#6e7781"}
fig["plants"] = bubble_timeline(plants, [(t, ENERGY_COLORS[t]) for t in ["水力", "地熱", "バイオマス", "陸上風力",
                                                                       "メガソーラー", "洋上風力"]],
                                "発電所の運転開始の年と出力")
g_top = dict(sorted(gaikoku.items(), key=lambda x: -max(x[1]))[:10])
fig["gaikoku"] = slope_chart(g_top, f"{G_OLD[:4]}年末", f"{G_NEW[:4]}年末", "国籍別の在留外国人数",
                             highlight={k for k, (a, b) in g_top.items() if b >= 2 * a and b >= 50})
SCHOOL_COLORS = {"保育施設": ORANGE, "小学校": GREEN, "中学校": BLUE, "義務教育学校": BLUE, "高校": RED,
                 "大学・専門学校": PURPLE}
school_order = ["保育施設", "小学校", "中学校", "高校", "大学・専門学校"]  # 小さい点から先に描く
fig["school_map"] = dot_map(
    [(*f["pos"], SCHOOL_COLORS[f["kind"]]) for k in school_order for f in located
     if (f["kind"] == k or (k == "中学校" and f["kind"] == "義務教育学校"))],
    [], "学校・保育施設の位置", r=3.6, opacity=0.85, edit=GSI_EDIT_SCHOOL, unit="1点 = 1か所",
    legend=[("保育施設(幼稚園・保育所など)", ORANGE), ("小学校", GREEN), ("中学校・義務教育学校", BLUE), ("高校", RED),
            ("大学・専門学校(一部)", PURPLE)]) if isj else ""
fig["kids_table"] = ratio_table(sorted(cities, key=lambda c: -kids[c]["小学校"]), STAGES,
                                lambda c, st: kids_per[c][st], pref_kids_per)
def name_list(title: str, names: list[str], by: dict, color: str) -> str:
    items = "".join(f"<li>{esc(n)} <small>({esc(by[n]['city'])}・{esc(by[n]['type'])})</small></li>" for n in names)
    return f'<div><h3 style="color:{color}">{esc(title)}({len(names)})</h3><ul class="names">{items}</ul></div>'


fig["nursery_types"] = ('<div class="two">' + name_list("一覧から消えた", closed, old_by, RED)
                        + name_list("新しく載った", opened, new_by, BLUE) + "</div>")
HYDRO_COLORS = ["#afb8c1", "#8c959f", ORANGE, GREEN, BLUE]
fig["hydro"] = line_chart(
    [(f"{fy}年度", HYDRO_COLORS[i % len(HYDRO_COLORS)], moving_avg(v)) for i, (fy, v) in enumerate(sorted(hydro.items()))],
    "県営水力発電所の1日の発電量", x_ticks=MONTH_START, x_fmt=lambda d: f"{MONTH_OF[int(d)]}月",
    y_fmt=lambda v: f"{v:,.0f}")
fig["hydro_month"] = hbar([(f"{m}月", by_month_hydro[m] / 1000) for m in [4, 5, 6, 7, 8, 9, 10, 11, 12, 1, 2, 3]],
                          "月ごとの発電量(5年の平均)", fmt=lambda v: f"{v:,.1f} GWh", color=GREEN,
                          highlight={"4月": BLUE, "5月": BLUE}, l=60)
CULTURE_COLORS = {"公民館など": "#8c959f", "図書館": BLUE, "博物館": PURPLE, "文化会館・ホール": RED}
fig["culture_map"] = dot_map(
    [(*c["pos"], CULTURE_COLORS[c["kind"]]) for k in CULTURE_COLORS for c in culture if c["kind"] == k],
    [], "博物館・図書館・公民館の位置", r=3.4, opacity=0.85, edit=GSI_EDIT_CULTURE, unit="1点 = 1か所",
    legend=[(f"{k}({culture_kinds[k]})", c) for k, c in CULTURE_COLORS.items()])
print(f"図 {len(fig)} 枚: {', '.join(fig)}")

In [ ]:
# ── 地震の想定震度の地図(250m の区画を PNG 1枚に塗って、地理院タイルの上に重ねる) ─────────────
# 区画が約 19 万あるので、点や四角を SVG で1つずつ描くと HTML が重くなる。区画を画素に塗った PNG を
# 標準ライブラリ(zlib)で作って data URI で埋め込む。地理院タイルと同じ Web メルカトルの画素に合わせる
import struct
import zlib

QUAKE_COLORS = [(0xdd, 0xe7, 0xee), (0xff, 0xe6, 0x96), (0xff, 0xc0, 0x40), (0xf5, 0x80, 0x20), (0xd0, 0x30, 0x20),
                (0x80, 0x00, 0x40)]  # 4以下 / 5弱 / 5強 / 6弱 / 6強 / 7
GSI_EDIT_QUAKE = "地理院タイルに地震の想定震度(250mメッシュ)と地名を追記して掲載"
RASTER_SCALE = 2  # ズーム TILE_Z の2倍の細かさで塗る(1区画がおよそ 4 x 5 画素)


def png_rgba(w: int, h: int, px: bytearray) -> bytes:
    raw = b"".join(b"\x00" + bytes(px[y * w * 4:(y + 1) * w * 4]) for y in range(h))
    chunk = lambda t, d: struct.pack(">I", len(d)) + t + d + struct.pack(">I", zlib.crc32(t + d))  # noqa: E731
    return (b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", struct.pack(">IIBBBBB", w, h, 8, 6, 0, 0, 0))
            + chunk(b"IDAT", zlib.compress(raw, 9)) + chunk(b"IEND", b""))


def mesh_raster(cells: list[tuple[str, tuple[int, int, int]]]) -> bytes:
    """(250m メッシュコード, 色) を AKITA_BBOX の範囲の PNG に塗る。"""
    x0, y0 = merc(AKITA_BBOX[0], AKITA_BBOX[3])
    x1, y1 = merc(AKITA_BBOX[2], AKITA_BBOX[1])
    W, H = math.ceil((x1 - x0) * RASTER_SCALE), math.ceil((y1 - y0) * RASTER_SCALE)
    px = bytearray(W * H * 4)
    for code, c in cells:
        lon, lat = mesh_sw(code)
        ax, ay = merc(lon, lat + 1 / 480)  # 北西の角
        bx, by = merc(lon + 1 / 320, lat)  # 南東の角
        rgba = bytes((*c, 215))
        for yy in range(max(0, int((ay - y0) * RASTER_SCALE)), min(H, math.ceil((by - y0) * RASTER_SCALE))):
            a = yy * W
            for xx in range(max(0, int((ax - x0) * RASTER_SCALE)), min(W, math.ceil((bx - x0) * RASTER_SCALE))):
                px[(a + xx) * 4:(a + xx) * 4 + 4] = rgba
    return png_rgba(W, H, px)


def raster_map(raster: bytes, labels: list[tuple[str, float, float]], label: str,
               legend: list[tuple[str, str]], edit: str, unit: str) -> str:
    """dot_map と同じ枠(地理院タイル・凡例・出典)で、点の代わりに PNG を1枚重ねる。"""
    global _basemap_defined
    x0, y0 = merc(AKITA_BBOX[0], AKITA_BBOX[3])
    x1, y1 = merc(AKITA_BBOX[2], AKITA_BBOX[1])
    w, h = x1 - x0, y1 - y0
    out = [svg_open(round(w), round(h), label).replace(f'viewBox="0 0 {round(w)} {round(h)}"',
                                                       f'viewBox="{x0:.1f} {y0:.1f} {w:.1f} {h:.1f}"'),
           f'<rect x="{x0:.1f}" y="{y0:.1f}" width="{w:.1f}" height="{h:.1f}" fill="#f6f8fa"/>']
    if basemap and not _basemap_defined:
        out.append('<g id="gsi-basemap">' + "".join(
            f'<image x="{tx * 256}" y="{ty * 256}" width="256" height="256" '
            f'href="data:image/png;base64,{base64.b64encode(body).decode()}"/>' for tx, ty, body in basemap) + "</g>")
        _basemap_defined = True
    elif basemap:
        out.append('<use href="#gsi-basemap"/>')
    gw = math.ceil(w * RASTER_SCALE) / RASTER_SCALE
    gh = math.ceil(h * RASTER_SCALE) / RASTER_SCALE
    out.append(f'<image x="{x0:.1f}" y="{y0:.1f}" width="{gw:.1f}" height="{gh:.1f}" preserveAspectRatio="none" '
               f'style="image-rendering:pixelated" href="data:image/png;base64,{base64.b64encode(raster).decode()}"/>')
    for name, lon, lat in labels:
        x, y = merc(lon, lat)
        out.append(f'<text x="{x:.1f}" y="{y:.1f}" font-size="14" font-weight="700" text-anchor="middle" '
                   f'fill="{INK}" stroke="#fff" stroke-width="3.5" paint-order="stroke">{esc(name)}</text>')
    for k, (name, c) in enumerate(legend):
        out.append(f'<rect x="{x0 + 10:.1f}" y="{y0 + 10 + 22 * k:.1f}" width="230" height="22" fill="#fff" fill-opacity="0.85"/>'
                   f'<rect x="{x0 + 16:.1f}" y="{y0 + 14 + 22 * k:.1f}" width="14" height="14" fill="{c}"/>'
                   f'<text x="{x0 + 36:.1f}" y="{y0 + 26 + 22 * k:.1f}" font-size="13" fill="{INK}">{esc(name)}</text>')
    lines = ([GSI_SOURCE, edit] if basemap else []) + [f"北が上 / {unit}"]
    for k, line in enumerate(reversed(lines)):
        out.append(f'<text x="{x1 - 8:.1f}" y="{y1 - 8 - 15 * k:.1f}" font-size="11" text-anchor="end" fill="{INK}" '
                   f'stroke="#fff" stroke-width="3" paint-order="stroke">{esc(line)}</text>')
    out.append("</svg>")
    return "".join(out)


if quake_mesh:
    hexc = lambda c: "#%02x%02x%02x" % c  # noqa: E731
    quake_png = mesh_raster([(code, QUAKE_COLORS[quake_class(m[0])]) for code, m in quake_mesh.items()])
    # 地名は、その市町村の区画の中央(中央値)に置く。役所の位置ではない
    q_city: dict[str, list[tuple[float, float]]] = defaultdict(list)
    for code, m in quake_mesh.items():
        q_city[m[2]].append(mesh_sw(code))
    q_labels = []
    for c in MAP_LABELS:
        if q_city.get(c):
            lons, lats = sorted(p[0] for p in q_city[c]), sorted(p[1] for p in q_city[c])
            q_labels.append((c, lons[len(lons) // 2], lats[len(lats) // 2]))
    fig["quake_map"] = raster_map(
        quake_png, q_labels, "27通りの想定のうち最も強い揺れ(震度)",
        [(f"震度{QUAKE_CLASSES[k][0]}  {man(quake_pop_cls[k])}人", hexc(QUAKE_COLORS[k])) for k in reversed(range(6))],
        GSI_EDIT_QUAKE, "1区画 = 250m 四方")
    QUAKE_SEA = "#0a3069"
    q_hl = {s["name"]: QUAKE_SEA for s in quake_scen if s["name"].startswith("海域")}
    fig["quake_pop6"] = hbar(sorted(((s["name"], s["pop6"]) for s in quake_scen), key=lambda x: -x[1]),
                             "震度6弱以上の地域に住む人(想定ごと)", fmt=lambda v: f"{man(v)}人", color=ORANGE,
                             highlight=q_hl, l=300)
    fig["quake_dead"] = hbar(sorted(((s["name"], s["dead"]) for s in quake_scen), key=lambda x: -x[1]),
                             "亡くなる人(想定ごと、冬の18時)", fmt=lambda v: f"{v:,.0f}人", color=RED,
                             highlight=q_hl, l=300)
    print(f"地震の地図: PNG {len(quake_png) / 1e3:,.0f} KB / 地名 {len(q_labels)}")
else:
    fig["quake_map"] = fig["quake_pop6"] = fig["quake_dead"] = ""

## 3. HTML を組み立てる

話題ごとに「ひとことで言うと」→ 図 → 補足 の順に並べる。数字は本文に埋め込むときもここで計算した値から作る
(手で書いた数字を置かない)。出典・ライセンス・加工したことを冒頭と末尾に、話題ごとに使ったデータセットを末尾に書く。

In [ ]:
from datetime import datetime, timedelta, timezone

JST = timezone(timedelta(hours=9))
now = datetime.now(JST).strftime("%Y-%m-%d %H:%M JST")
wind = sum(e["kw"] for e in energy if "風力" in e["label"]) / 1000
wind_share = wind * 1000 / sum(e["kw"] for e in energy)
fewest_birth_ratio = births[y1] / births[min(births)]
young_share = young / out_total
work_share = sum(out_by_age[a]["転職・就職"] + out_by_age[a]["就職(学生)"] for a in AGE_GROUPS[:2]) / \
    sum(sum(out_by_age[a].values()) for a in AGE_GROUPS[:2])
study_1519 = out_by_age["15〜19歳"]["学業"] / sum(out_by_age["15〜19歳"].values())
kuma_prev = bear_year.get(top_year - 1, 0)
kuma_peak_month = max(range(1, 13), key=lambda m: by_month[top_year][m])
smallest_drop = max(city_change, key=city_change.get)
largest_drop = min(city_change, key=city_change.get)
oldest = max(city_aged, key=city_aged.get)
youngest = min(city_aged, key=city_aged.get)
umbrella = found_items.get("かさ類", 0)
offshore = load("050008_shigen-ene_003", r"洋上風力", "発電設備の出力")
ERA_BASE = {"R": 2018, "H": 1988, "S": 1925}
offshore_start = "、".join(sorted({f"{ERA_BASE[m.group(1)] + int(m.group(2))}年{int(m.group(3))}月"
                                   for r in offshore if len(r) > 1
                                   for m in [re.match(r"([RHS])(\d+)\.(\d+)", unicodedata.normalize("NFKC", r[1]).strip())] if m}))


def safe_url(u: str | None) -> str:
    return esc(u) if u and u.startswith("https://ckan.pref.akita.lg.jp/") else ""


_no = iter(range(1, 100))


def part(title: str, lead: str) -> str:
    return f'<div class="part"><p class="part-title">{esc(title)}</p><p class="sub">{esc(lead)}</p></div>'


def section(title: str, lead: str, body: str, note: str = "") -> str:
    return (f'<section><h2><span class="no">{next(_no)}</span>{esc(title)}</h2><p class="lead">{lead}</p>{body}'
            + (f'<p class="note">{note}</p>' if note else "") + "</section>")


sources = "".join(
    f'<li><a href="{safe_url((ds_meta.get(ds) or {}).get("dataset_url"))}" rel="noopener">'
    f'{esc((ds_meta.get(ds) or {}).get("dataset_title", ds))}</a>({esc((ds_meta.get(ds) or {}).get("organization") or "")})'
    f' ― {esc("、".join(purposes))}</li>' for ds, purposes in used.items())

sections = [
    part("人と人口", "人口の減り方、年齢、県内外への移動"),
    section(f"{y1 - y0}年で、人口は{round((1 - pop[y1] / pop[y0]) * 10)}割減った",
            f"秋田県の人口は {y0} 年の <b>{pop[y0] / 1e4:,.1f} 万人</b> から、{y1} 年には <b>{pop[y1] / 1e4:,.1f} 万人</b> になった"
            f"({(pop[y1] / pop[y0] - 1) * 100:+.0f}%)。この1年だけでも {abs(pop[y1] - pop[y1 - 1]):,.0f} 人減っている。",
            fig["pop"], "各年10月1日現在の推計人口(国勢調査の年は確定値)。"),
    section("生まれる人より、亡くなる人のほうが多い",
            f"{cross} 年に亡くなった人が生まれた人を上回り、その差は広がり続けている。{y1} 年は生まれた人 <b>{births[y1]:,.0f} 人</b>に対して、"
            f"亡くなった人 <b>{deaths[y1]:,.0f} 人</b>。生まれた人は {min(births)} 年の {fewest_birth_ratio * 100:.0f}% まで減った。"
            f"人口が減る理由の多くはこちら(自然減)で、県外への転出超過(社会減)は {y1} 年で {abs(social[y1]):,.0f} 人。",
            fig["births"], "1年間(前年10月〜その年の9月)の出生者数・死亡者数。"),
    section("人口の形は「ピラミッド」から「つぼ」へ",
            f"1歳ごとの人数を、男性を左・女性を右に並べた。線が 1980 年、塗りが 2025 年。いちばん人数が多い年齢は、"
            f"1980 年の <b>{peak(pyr_old)} 歳</b>から 2025 年の <b>{peak(pyr_new)} 歳</b>に移った。若い世代の幅が大きく細くなっている。",
            fig["pyramid"], "100歳以上はまとめて1本。年齢が分からない人は含めていない。"),
    section(f"県民の{pref_aged[CITY_TO]:.0f}%が65歳以上",
            f"65歳以上の割合は 1980 年の {pref_aged[1980]:.1f}% から {CITY_TO} 年は <b>{pref_aged[CITY_TO]:.1f}%</b> になった。",
            fig["aged_trend"]),
    section("市町村ごとに見ると",
            f"{CITY_FROM} 年から {CITY_TO} 年の15年間で、県全体の人口は {pref_change * 100:+.0f}%。いちばん減り方が小さいのは "
            f"<b>{esc(smallest_drop)}</b>({city_change[smallest_drop] * 100:+.0f}%)、大きいのは <b>{esc(largest_drop)}</b>"
            f"({city_change[largest_drop] * 100:+.0f}%)。65歳以上の割合は <b>{esc(youngest)}</b> の {city_aged[youngest]:.1f}% から "
            f"<b>{esc(oldest)}</b> の {city_aged[oldest]:.1f}% まで開きがある。",
            ((f'<div class="two maps"><div><h3>人口({CITY_TO}年)</h3>{fig["pop_map"]}</div>'
              f'<div><h3>人口の増減({CITY_FROM}年→{CITY_TO}年)</h3>{fig["change_map"]}</div></div>'
              '<p class="sub">パソコンでは、市町村にマウスを乗せると人口・増減・65歳以上の割合が出る。</p>')
             if fig["pop_map"] else "")
            + '<h3>人口の増減(' + f"{CITY_FROM}年→{CITY_TO}年" + ')</h3>' + fig["city_change"]
            + f'<h3>65歳以上の割合({CITY_TO}年)</h3>' + fig["city_aged"],
            "秋田市は青。市町村の比較は合併で今の25市町村がそろった後の 2010 年から。"),
    section(f"{age_worst[0]}歳と{age_worst[1]}歳で、県外へ",
            f"1歳ごとに、県外から移ってきた人から県外へ移った人を引いた({kakusai_year[:4]}年の1年間)。"
            f"<b>{age_worst[0]}歳</b>({age_net[age_worst[0]]:+,.0f}人)と <b>{age_worst[1]}歳</b>({age_net[age_worst[1]]:+,.0f}人)で大きく減る。"
            f"高校を出て進学・就職する年齢と、大学などを出て就職する年齢に当たる。全年齢では {age_net_total:+,.0f} 人。",
            fig["age_net"], "青は転入が多い年齢、赤は転出が多い年齢。県内での移動は含めていない。91歳以上は省いた。"),
    section("若い人は、仕事と進学で県外へ",
            f"県外へ転出した人に理由を聞いた調査({esc(riyu_year)})。15〜29歳が回答者の <b>{young_share * 100:.0f}%</b> を占める。"
            f"15〜19歳は「学業」が {study_1519 * 100:.0f}%、15〜24歳では「転職・就職」と「就職(学生)」を合わせて {work_share * 100:.0f}% になる。",
            fig["reasons"], "回答した人の数による割合で、転出した人全員ではない。年齢不詳と15歳未満は除いた。"),
    section("人が動くのは3月",
            f"1か月ごとの社会増減(転入 − 転出)を、{sm_years[0]}年から{sm_years[-1]}年まで並べた。毎年 <b>3月</b>に赤く(転出超過)、"
            f"4月に青く(転入超過)なる。直近 {len(recent)} 年の平均は 3月 <b>{avg_month[3]:+,.0f}人</b>、4月 {avg_month[4]:+,.0f}人。"
            "3月に出ていく人の多くは4月に戻ってこない。",
            '<div class="scroll">' + fig["social_month"] + "</div>",
            "各月1日現在の推計人口にある「社会動態」を、その前の月の1か月分として置いた。"),
    section("県内では、秋田市へ",
            f"県内の市町村どうしの引っ越しのうち、秋田市との出入りだけを取り出した({esc(idou_year[:4])}年の1年間)。"
            f"秋田市は {sum(1 for v in akita_net.values() if v > 0)} の市町村から転入のほうが多く、県内だけで "
            f"<b>{akita_gain:+,.0f}人</b>。",
            fig["akita_pull"], "棒は「その市町村から秋田市へ移った人 − 秋田市からその市町村へ移った人」。赤は秋田市から出ていくほうが多い市町村。"),
    part("子どもと学校", "学校・保育施設の場所と、子どもの数"),
    section("学校と保育施設の場所",
            f"保育施設 {count_of(None, '保育施設')} か所、小学校 {sum(1 for f in facilities if f['kind'] == '小学校')} 校、"
            f"中学校 {sum(1 for f in facilities if f['kind'] == '中学校')} 校、義務教育学校 {sum(1 for f in facilities if f['kind'] == '義務教育学校')} 校、"
            f"高校 {count_of(None, '高校')} 校を地図に置いた。<b>高校が無い市町村は {len(no_high)}</b>"
            f"({esc('、'.join(no_high))})。",
            ('<div class="map">' + fig["school_map"] + "</div>") if fig["school_map"] else '<p class="sub">(位置参照情報を取れなかったので地図は無し)</p>',
            f"位置は住所の大字・町丁目の代表点で、建物の位置ではない。同じ大字・町丁目に複数あるときは、代表点のまわりに少しずらして置いた。住所を照らし合わせられなかった {len(facilities) - len(located)} か所"
            "(「字○○」が位置参照情報に無いものなど)は載せていない。大学・専門学校は「確認大学等一覧」(高等教育の修学支援新制度の対象)の"
            f"{sum(1 for f in facilities if f['kind'] == '大学・専門学校')} 校だけで、国立大学や私立大学などは入っていない。"
            "特別支援学校は含めていない。"),
    section("子ども何人に、学校が1つ",
            f"市町村ごとに、その年齢の子どもの数を学校・保育施設の数で割った({esc(kids_year[:4])}年10月1日の人口)。"
            f"県全体では、小学生の年齢(6〜11歳)の子ども <b>{pref_kids_per['小学校']:,.0f}人に小学校が1校</b>、"
            f"15〜17歳 {pref_kids_per['高校']:,.0f}人に高校が1校。数が大きい(色が濃い)ほど、1つの学校・施設に多くの子どもがいる。",
            '<div class="scroll">' + fig["kids_table"] + "</div>",
            "小学校・中学校には義務教育学校を含めた。私立・国立の学校も含み、学区や通学先は考えていない(高校は市町村をまたいで通うことが多い)。"
            "6〜11歳の子どもが多い順。"),
    section(f"保育施設は1年で{len(nursery_old) - len(nursery_new)}か所減った",
            f"幼稚園・保育所・認定こども園の一覧を1年分比べた。<b>{len(nursery_old)} か所から {len(nursery_new)} か所</b>になった"
            f"(保育所 {type_old['保育所']} → {type_new['保育所']}、認定こども園 {type_old['認定こども園']} → {type_new['認定こども園']}、"
            f"幼稚園 {type_old['幼稚園']} → {type_new['幼稚園']})。一覧から名前が消えた施設は {len(closed)}、新しく載った名前は {len(opened)}。",
            fig["nursery_types"],
            "2025年4月1日現在と2026年4月1日現在の一覧の比較。名前が変わっただけの施設も「消えた」「新しく載った」に数えている。"),
    part("自然とエネルギー", "クマ、発電所、水力の季節"),
    section(f"{top_year} 年、クマの出没が急に増えた",
            f"クマ出没情報システム「クマダス」に記録されたツキノワグマの出没(目撃・痕跡・人身被害)は、{top_year} 年に "
            f"<b>{bear_year[top_year]:,} 件</b>。前の年({kuma_prev:,} 件)の {bear_year[top_year] / kuma_prev:.0f} 倍になった。"
            f"最も多かったのは {kuma_peak_month} 月で {by_month[top_year][kuma_peak_month]:,} 件。人身被害の記録は {top_year} 年に "
            f"{injury_year.get(top_year, 0)} 件。",
            '<h3>月ごとの件数</h3>' + fig["kuma_month"]
            + f'<h3>{top_year}年の出没地点</h3><div class="map">' + fig["kuma_map"] + "</div>",
            f"記録は {kuma_first:%Y年%-m月} から {kuma_last:%Y年%-m月%-d日} まで。{kuma_first.year} 年は {kuma_first.month} 月から、"
            f"{kuma_last.year} 年は {kuma_last.month} 月までなので、その前後の月は線を引いていない。"
            "件数は通報・投稿などで記録された数で、クマの数そのものではない(同じクマが何度も記録されることもある)。"
            + (f'背景の地図は<a href="{GSI_TILES_URL}" rel="noopener">国土地理院の地理院タイル</a>(淡色地図)。その上にクマの出没地点と地名を書き足した。' if basemap else "") + f"太字の地名は、その市町村で記録された出没地点の中央に置いた目安。{top_year} 年に多かった市町村: "
            + esc("、".join(f"{c} {n:,}件" for c, n in kuma_city.most_common(5))) + "。"),
    section("クマが出るのは、朝と夕方",
            f"出没の記録を時刻と月で数えた。<b>朝 5〜8時</b>({dawn * 100:.0f}%)と<b>夕方 16〜19時</b>({dusk * 100:.0f}%)に多く、"
            "秋(9〜11月)はその山がさらに大きくなる。",
            fig["kuma_hour"], f"{kuma_first:%Y年%-m月}から{kuma_last:%Y年%-m月}までの合計。"
            "時刻は記録された目撃日時のもので、ちょうどの時刻(6:00 など)に丸めて記録されたものが多い。"),
    section("風の県、秋田",
            f"県内の主な発電設備の出力を種類ごとに足すと、風力(陸上+洋上)が <b>{wind:,.0f} MW</b> で、ここに挙げた設備の "
            f"{wind_share * 100:.0f}% を占める。能代港・秋田港の港湾区域では、洋上風力が {offshore_start} に運転を始めた。",
            fig["energy"],
            "表ごとに調べた時期が違う(" + esc("、".join(f"{e['label']} {e['asof']}" for e in energy))
            + ")。バイオマスには石炭などとの混焼の設備を含む。最大出力の合計で、実際に発電した量ではない。"),
    section("明治の水力から、令和の洋上風力へ",
            f"表に載っている {len(plants)} か所の発電所を、運転を始めた年に並べた。最も古いのは <b>{oldest_plant['year']}年</b>"
            f"({esc(oldest_plant['name'])}、{esc(oldest_plant['type'])})。地熱は {by_type_first['地熱']}年、陸上風力は {by_type_first['陸上風力']}年、"
            f"メガソーラーは {by_type_first['メガソーラー']}年、洋上風力は {by_type_first['洋上風力']}年から。",
            fig["plants"], "運転開始は表の和暦(M30.6 など)の年。リプレースで撤去された設備(表では負の値)は除き、建て替え後のものを数えた。"),
    section("雪どけの春に、水力はいちばん働く",
            f"県営の水力発電所 {n_hydro_plants} か所の1日の発電量を、{min(hydro)}〜{max(hydro)}年度の5年分重ねた(7日間の平均)。"
            f"毎年 <b>4〜5月</b>に山ができ、この2か月で1年の <b>{spring_share * 100:.0f}%</b> を発電する。"
            f"山に積もった雪がとけて川の水が増えるため。いちばん少ないのは {hydro_low}月。",
            fig["hydro"] + f"<h3>月ごとの発電量({min(hydro)}〜{max(hydro)}年度の平均)</h3>" + fig["hydro_month"],
            "県営(秋田県企業局)の発電所だけで、県内の水力発電の一部。点検などで止まっている日は0になる。"),
    part("まちと文化", "博物館・図書館・公民館"),
    section("博物館・図書館・公民館",
            f"社会教育施設 {len(culture)} か所を地図に置いた。公民館などが {culture_kinds['公民館など']}、図書館が {culture_kinds['図書館']}、"
            f"博物館(博物館に類する施設を含む)が {culture_kinds['博物館']}、文化会館・ホールが {culture_kinds['文化会館・ホール']}。"
            f"図書館はすべての市町村にあるわけではなく、<b>{sum(1 for v in lib_per.values() if not v)} の市町村</b>には一覧に図書館が無い。",
            '<div class="map">' + fig["culture_map"] + "</div>",
            "位置は一覧に載っている緯度・経度。図書館は「生涯学習・社会教育関係」、博物館と文化会館は「文化・芸術関係」の一覧から。"
            "公民館などには、公民館類似施設・青少年施設・生涯学習センターなどを含めた。"),
    *([part("地震への備え", "県の地震被害想定(27 通りの地震)から"),
       section(f"どれかの想定で、{quake_pop6 / quake_pop * 100:.0f}%の人が震度6弱以上の揺れに",
               f"秋田県の地震被害想定は、県内外の断層や日本海の地震など <b>27 通り</b>の地震について、250m 四方の区画ごとに揺れや被害を計算している。"
               f"区画ごとに、{'それら' if not quake_missing else f'そのうち使えた {len(quake_scen)} 通り'}の中で最も強い揺れの震度で塗ると、"
               f"<b>震度6弱以上</b>になる区画に住む人は <b>{man(quake_pop6)}人</b>"
               f"(想定の人口 {man(quake_pop)}人の {quake_pop6 / quake_pop * 100:.0f}%)。震度7 になる区画にも {man(quake_pop_cls[5])}人が住んでいる。",
               '<div class="map">' + fig["quake_map"] + "</div>"
               + f"<p>想定ごとに見ると、震度6弱以上の地域に住む人が最も多いのは <b>{esc(quake_widest['name'])}</b>({man(quake_widest['pop6'])}人)、"
               f"亡くなる人が最も多いのは <b>{esc(quake_worst['name'])}</b>({quake_worst['dead']:,.0f}人、冬の18時)。</p>"
               + "<h3>震度6弱以上の地域に住む人(想定ごと)</h3>" + fig["quake_pop6"]
               + "<h3>亡くなる人(想定ごと、冬の18時)</h3>" + fig["quake_dead"],
               "地図は想定ごとの揺れの最大を区画ごとに重ねたもので、これらの地震が一度に起きるわけではない。"
               + "".join(f"「{no:02d} {esc(name)}」は公開されている CSV の中に文字コードとして読めない箇所があり、bronze に入れていないので除いた。"
                         for no, (name, _) in sorted(quake_missing.items()))
               + "調査は平成25年(2013年、ファイル名の h25 から)で、人口・建物はその時点のもの。"
               f"陸の区画のうち計測震度が空の {quake_scen[0]['no_int']:,} 区画(人口 {quake_scen[0]['no_int_pop']:,.0f}人)は塗っていない。"
               "亡くなる人は CSV の「死者数(冬18時)」の合計(建物の倒壊・急傾斜地の崩壊・火災)で、このデータには津波による死者の列が無い。"
               "棒の紺色は日本海の海域の地震。"
               + (f'背景の地図は<a href="{GSI_TILES_URL}" rel="noopener">国土地理院の地理院タイル</a>(淡色地図)。その上に想定震度と地名を書き足した。' if basemap else "")
               + "太字の地名は、その市町村の区画の中央に置いた目安。")]
      if quake_mesh else []),
    part("暮らし", "落とし物、県内に暮らす外国人"),
    section(f"落とし物は年に{found[lost_year][0] / 1e4:.0f}万件",
            f"{lost_year} 年に警察へ届けられた落とし物は <b>{found[lost_year][0]:,.0f} 件</b>、現金は約 <b>{found[lost_year][1] / 1e5:,.1f} 億円</b>。"
            f"持ち主に返った現金は約 {returned[lost_year][1] / 1e5:,.1f} 億円。物では証明書・カードが最も多く、かさも {umbrella:,.0f} 本届いている。",
            fig["lost_items"], "「その他の物品」は除いた。"),
    section("県内に暮らす外国人",
            f"{esc(G_NEW[:4])}年末に県内に住む外国人は <b>{gaikoku_total:,.0f}人</b>。{esc(G_OLD[:4])}年末からの5年で、"
            f"<b>{esc(fastest)}</b>は {gaikoku[fastest][0]:,.0f}人から {gaikoku[fastest][1]:,.0f}人に増えた。",
            fig["gaikoku"], "人数の多い10か国・地域。縦は対数で、線の傾きが同じなら増え方(倍率)が同じ。2倍以上に増えたものを赤で示した。住民基本台帳(各年12月31日)による。"),
]

CSS = """
body{font-family:-apple-system,BlinkMacSystemFont,"Hiragino Sans","Noto Sans JP","Yu Gothic",Meiryo,sans-serif;
max-width:860px;margin:0 auto;padding:28px 18px 72px;color:#1f2328;line-height:1.8;font-size:17px}
h1{font-size:1.9rem;line-height:1.35;margin:.2em 0 .1em}h2{font-size:1.35rem;margin:2.4em 0 .3em;line-height:1.45}
h3{font-size:1rem;margin:1.4em 0 .2em;color:#59636e}
.no{display:inline-block;background:#1f6feb;color:#fff;border-radius:50%;width:1.6em;height:1.6em;line-height:1.6em;
text-align:center;font-size:.85em;margin-right:.5em}
.lead b{color:#d1242f}.note,.sub,small{color:#59636e;font-size:.85rem}a{color:#0969da}
.kpi{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:12px;margin:22px 0}
.kpi div{background:#f6f8fa;border:1px solid #d0d7de;border-radius:10px;padding:12px 16px;font-size:.9rem;color:#59636e}
.kpi b{display:block;font-size:1.7rem;color:#1f2328;line-height:1.3}
.map{max-width:560px}table.ratio{border-collapse:collapse;font-size:.88rem;margin:8px 0}table.ratio th,table.ratio td{border:1px solid #d0d7de;padding:3px 10px;text-align:right}table.ratio thead th{text-align:center;background:#f6f8fa}table.ratio tbody th,table.ratio tfoot th{text-align:left;font-weight:400}table.ratio td.none{color:#d1242f;text-align:center;background:#fff8f8}table.ratio tfoot td,table.ratio tfoot th{background:#f6f8fa}.two{display:grid;grid-template-columns:1fr 1fr;gap:16px}.choro path{cursor:default}.choro path:hover{stroke:#1f2328;stroke-width:2.5;paint-order:stroke}@media (max-width:640px){.two.maps{grid-template-columns:1fr}}ul.names{margin:.2em 0;padding-left:1.2em;font-size:.92rem}.scroll{overflow-x:auto}
.part{margin:3.2em 0 0;padding:14px 18px;background:#ddf4ff;border-radius:10px}.part-title{font-size:1.2rem;font-weight:700;margin:0;color:#0a3069}.part .sub{margin:.2em 0 0}section{border-top:1px solid #d0d7de}
footer{margin-top:3em;font-size:.85rem;color:#59636e;border-top:1px solid #d0d7de;padding-top:1em;line-height:1.7}
"""

page = f"""<!doctype html>
<html lang="ja"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<meta http-equiv="Content-Security-Policy" content="default-src 'none'; style-src 'unsafe-inline'; img-src data:">
<title>{esc(TITLE)}</title><style>{CSS}</style></head><body>
<h1>{esc(TITLE)}</h1>
<p class="sub">作成 {esc(now)} / データ取得 {esc(manifest['landing_fetched_at'])}(UTC)</p>
<p>秋田県が公開しているオープンデータから、人口・移動・学校・クマ・エネルギー・文化・落とし物・外国人など、暮らしに近い数字と、地震の被害想定を選んで図にした。</p>
<p class="sub">出典: <a href="https://ckan.pref.akita.lg.jp/" rel="noopener">秋田県オープンデータカタログサイト</a>(秋田県)、
<a href="https://creativecommons.org/licenses/by/4.0/deed.ja" rel="license noopener">CC BY 4.0</a>。
このページは同サイトのデータを加工(集計、図の作成)して作成したもので、秋田県が作成したものではない。</p>
<div class="kpi">
<div><b>{pop[y1] / 1e4:,.1f}万人</b>人口({y1}年)</div>
<div><b>{births[y1]:,.0f}人</b>生まれた人({y1}年)</div>
<div><b>{pref_aged[CITY_TO]:.1f}%</b>65歳以上の割合</div>
<div><b>{bear_year[top_year]:,}件</b>クマの出没({top_year}年)</div>
<div><b>{wind:,.0f} MW</b>風力発電の出力</div>
</div>
{''.join(sections)}
<footer>
<p><b>使ったデータ</b>(いずれも秋田県、CC BY 4.0)</p><ul>{sources}</ul>
{f'<p>地図(クマ、学校・保育施設、博物館・図書館{"、地震" if quake_mesh else ""}): {esc(GSI_SOURCE)}(<a href="{GSI_TILES_URL}" rel="noopener">地理院タイル一覧</a>)。{esc(GSI_EDIT)}。{esc(GSI_EDIT_SCHOOL)}。{esc(GSI_EDIT_CULTURE)}。{esc(GSI_EDIT_QUAKE) + "。" if quake_mesh else ""}国土地理院が作成したものではない。</p>' if basemap else ''}
{f'<p>市町村の境界: {esc(N03_SOURCE)}(<a href="{N03_PAGE}" rel="noopener">国土数値情報 行政区域データ</a>、CC BY 4.0)。</p>' if city_paths else ''}
{f'<p>学校・保育施設の位置: {esc(ISJ_SOURCE)}を使い、住所の大字・町丁目の代表点に置いた(編集・加工: 本ページの作成者)。</p>' if isj else ''}
<p>{esc(TITLE)} / 本ページは出典のデータを加工して作成した。数値はデータの公開時点のもの。</p>
<p>JAWS FESTA 2026 in AKITA「CSV置くだけ構成は壊れずに間違う ― 秋田のオープンデータで検証するS3 Tables」の検証の一部として作成。</p>
</footer>
</body></html>
"""
body = page.encode("utf-8")
print(f"HTML: {len(body) / 1e6:.2f} MB / 使ったデータセット {len(used)}")

In [ ]:
# ── 公開してはいけない情報が入っていないか(バケット名・アカウント ID) ─────────
leaks = [s for s in {getattr(store, "bucket", None), globals().get("account_id"), str(getattr(store, "root", ""))}
         if s and s in page]
assert not leaks, f"HTML に公開してはいけない文字列が入っている: {leaks}"
assert not re.search(r"<(script|link|iframe|img)\b", page, re.I), "外部を読み込む要素がある"
assert all(safe_url((ds_meta.get(ds) or {}).get("dataset_url")) for ds in used), "出典のリンクが無いデータセットがある"
if basemap:  # 地図を埋め込んだら、出典・加工の記載・リンクがそろっていること
    assert all(x in page for x in (GSI_SOURCE, GSI_EDIT, GSI_EDIT_SCHOOL, GSI_EDIT_CULTURE, GSI_TILES_URL)), "地理院タイルの出典の記載が足りない"
    assert not quake_mesh or GSI_EDIT_QUAKE in page, "地震の地図の加工の記載が足りない"
if city_paths:  # 行政区域データを使ったら、出典と加工したことがあること(CC BY 4.0)
    assert N03_SOURCE in page and N03_PAGE in page, "行政区域データの出典の記載が足りない"
if isj:  # 位置参照情報を使ったら、利用約款どおりの出典(名称・整備年・ファイル名)があること
    assert ISJ_SOURCE in page, "位置参照情報の出典の記載が足りない"
print("公開前の確認 OK")

In [ ]:
# ── 書き出す ────────────────────────────────────────────────────────────────
if LOCAL_OUT:
    Path(LOCAL_OUT).mkdir(parents=True, exist_ok=True)
    Path(LOCAL_OUT, "index.html").write_bytes(body)
    print(f"手元: {Path(LOCAL_OUT).resolve()}/index.html")
if DRY_RUN:
    print(f"[dry-run] {OUTPUT_PREFIX}index.html は書かない")
else:
    store.put(OUTPUT_PREFIX + "index.html", body, "text/html; charset=utf-8")
    print(f"書いた: {store.label}{OUTPUT_PREFIX}index.html")

In [ ]:
# ノートブック上で見る(JupyterLab ではこのセルの出力にページが表示される)
from IPython.display import HTML, display

display(HTML(f'<iframe srcdoc="{esc(page)}" style="width:100%;height:900px;border:1px solid #ccc"></iframe>'))